<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-07/NB07_vision_transformer.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html)

# Week 07: Vision Transformer (ViT): The new generation in image processing

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lab.html) · [PDF notes](Week07_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-07/NB07_vision_transformers.ipynb)

</div>

**This notebook has two parts.** In the first, an artificial intelligence (AI) assistant writes the code from a prompt, a written instruction. In the second, you write the code yourself in Python. Both parts reach the same results with the same data, so choose one of them at each practice stop of the lecture page.

**Together with the lecture page.** You come to this Jupyter notebook from the practice stops of the lecture page. At the end of every stop, a link takes you back to the lecture page, to the place where reading continues.

| | For whom? | What you do |
|---|---|---|
| **[Part 1: Prompt path](#part-1)** | Participants who do not write code | An AI assistant writes the code from ready prompts; you run it and check the result. |
| **[Part 2: Python path](#part-2)** | Participants who want to learn Python | You write and run the code step by step, with exercises. |

**Where it runs.** The same notebook runs in Google Colab, with nothing to install, and in Jupyter on your own computer (Anaconda; see SETUP.md in the course folder). Choose the one you prefer. In Colab, save your own copy first with *File → Save a copy in Drive*; otherwise the changes are lost when the tab is closed. Training each small network of Sections 5 to 7 takes one to three minutes on an ordinary laptop, the features of the 600 echocardiography frames take about two minutes and the mini ViT of the mini project in Section 4 about five minutes on CIFAR-10; no graphics card is needed, and in Colab a graphics card (Runtime, Change runtime type) shortens all of it. Run Section 0 first, whichever part you choose. It prepares the course files and the packages. A link of the lecture page opens the notebook at a stop or a section: There, run the first cell, the preparation cell or the catch-up cell. It also runs Section 0 if this has not happened in the session.

**Data and privacy.** Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution and with the approval of an ethics committee, and never in a cloud notebook or in an AI chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week. The second cell downloads what is missing: the apical echocardiograms of 300 patients of CAMUS with their expert outlines (about 227 MB, already there if Week 4 ran on this computer) from the public database of the Human Heart Project (CREATIS, Lyon), the ImageNet weights of ViT-B/16 (about 330 MB) and of EfficientNet-B0 (about 21 MB), and the stenosis set of ARCADE from Zenodo (an archive of 451.6 MB, of which only the stenosis set is kept). The first run takes a few minutes; later runs skip the files that are already there. CAMUS is free for non-commercial scientific research under CC BY-NC-SA 4.0, and every use of CAMUS cites Leclerc et al., IEEE Transactions on Medical Imaging, 2019. ARCADE is published under CC0, and its paper is Popov et al., Scientific Data, 2024. The synthetic angiograms of the virtual clinic are drawn in the notebook. Two downloads come later, when their cells run: the ready vision transformer of the Hugging Face Hub in Section 3 (about 350 MB) and CIFAR-10 for the mini project in Section 4 (about 170 MB from the University of Toronto).

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy

from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import angiogram, arcade, camus, coronary

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import vit_b_16, ViT_B_16_Weights, efficientnet_b0, EfficientNet_B0_Weights
except ImportError as error:
    raise ImportError("PyTorch or torchvision is not installed. Install them as SETUP.md describes, then restart the kernel.") from error
try:
    import skimage
except ImportError as error:
    raise ImportError("scikit-image is not installed. Install it with pip install scikit-image, then restart the kernel.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 8, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
camus.download()
arcade.download()
ViT_B_16_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
EfficientNet_B0_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
print("ImageNet weights of ViT-B/16 and EfficientNet-B0 are ready.")
print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 7.1": (196, 0),
    "Exercise 7.2": (0.284, 0.001),
    "Exercise 7.3": (0.685, 0.001),
    "Exercise A": (274, 0),
    "Exercise B": (0.665, 0.001),
    "Exercise C": (7087872, 0),
    "Exercise D": (0.885, 0.01),
    "Exercise E": (0.176, 0.001),
    "Exercise F": (1.625, 0.001),
    "Exercise G": (133888, 0),
    "Exercise H": (0.139, 0.02),
    "Exercise I": (0.68, 0.05),
    "Exercise J": (905, 25),
    "Step 1.1": (0.704, 0.002),
    "Step 1.2": (0.862, 0.01),
    "Step 1.3": (0.782, 0.01),
    "Step 1.4": (0.925, 0.05),
    "Step 2.1": (0.965, 0.05),
    "Step 2.2": (0.538, 0.02),
    "Step 2.3": (0.81, 0.05),
    "Step 3.1": (0.93, 0.05),
    "Step 3.2": (0.934, 0.05),
    "Step 3.3": (0.153, 0.01),
})
print("Expected answers are ready.")

In [ ]:
#@title Tables of the exercise prompts (run this cell; the Python path does not need to read it)
def exercise_table(key):
    """The table "data" of the prompt of an exercise, built from the data of the week."""
    g = globals()
    k = key
    sets = {"training": "training", "validation": "validation", "test": "test", "testing": "test", "train": "training", "val": "validation"}
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Run the sections before this exercise first."))
    if k in ("7.1", "7.3"):
        left, truth = angiogram.patient_frames("VC-2427")["left"]
        if k == "7.1":
            return pd.DataFrame({"angiogram": [np.asarray(Image.fromarray(left).resize((224, 224), Image.BILINEAR)) / 255]})
        s = truth["stenoses"][1]
        return pd.DataFrame({"angiogram": [left / 255], "x": [int(s["x"])], "y": [int(s["y"])]})
    if k == "A":
        pats, les = angiogram.registry()
        t = les.merge(pats[["patient_id", "split"]], on="patient_id")
        t = t[t["split"] == "test"]
        return pd.DataFrame({"patient": t["patient_id"].to_numpy(), "vessel": t["vessel"].to_numpy(), "stenosis": t["stenosis"].to_numpy(),
                             "view": ["left" if angiogram.VIEW_OF[v] == "left" else "right" for v in t["vessel"]],
                             "significant": [int(coronary.significant(a, b)) for a, b in zip(t["territory"], t["stenosis"])]})
    if k == "D":
        echo, echo_split, echo_patient, echo_view, echo_quality = need("echo", "echo_split", "echo_patient", "echo_view", "echo_quality")
        q = {"Good": "good", "Medium": "medium", "Poor": "poor"}
        return pd.DataFrame({"set": [sets[s] for s in echo_split], "patient": echo_patient, "view": echo_view,
                             "quality": [q[x] for x in echo_quality], "frame": list(echo)})
    if k == "E":
        (syn,) = need("syn")
        return pd.concat([pd.DataFrame({"set": sets[s], "patient": syn[s][2], "stenosis": syn[s][1]}) for s in syn], ignore_index=True)
    if k == "F":
        return pd.DataFrame([{"set": sets[s], "angiogram": a["id"], "stenoses": len(a["stenoses"])}
                             for s in arcade.SPLITS for a in arcade.annotations(s)])
    if k == "H":
        maps_te, yt, ARC = need("maps_te", "yt", "ARC")
        return pd.DataFrame({"angiogram": ARC["test"][2], "stenosis": yt, "attention": [m / m.sum() for m in maps_te]})
    if k == "I":
        cnn_arcade, thresholds, test_ann, heat_map = need("cnn_arcade", "thresholds", "test_ann", "heat_map")
        return pd.DataFrame({"angiogram": [a["id"] for a in test_ann],
                             "probabilities": [heat_map(cnn_arcade, arcade.load("test", a["file"]))[0] for a in test_ann],
                             "threshold": thresholds["CNN"]})
    if k == "J":
        (ARC,) = need("ARC")
        return pd.DataFrame({"angiogram": ARC["train"][2]})
    raise KeyError(k)
print("The tables of the exercise prompts are ready.")

<a id="part-1"></a>

# Part 1: Prompt path

**For whom?** This part is for participants who do not write code. Here you type no code. An artificial intelligence (AI) assistant writes it from a ready prompt, a written instruction; you run the code and check the result.

**What you need.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab: Open it with the Gemini button at the top of the window. ChatGPT, Claude, Copilot or Gemini in another browser tab work as well.

**How a step works.**

1. Read the goal of the step.
2. Run the data cell under the prompt. It prepares the table "data" that the prompt describes and shows its first and last rows.
3. Copy the prompt: Select it with the mouse and press Ctrl+C (Cmd+C on a Mac).
4. Paste it into the AI assistant and send it.
5. Copy the code that the assistant returns, paste it into the cell under the data cell in place of the line that starts with `#`, and run the cell with the ▶ button or with Shift+Enter.
6. Run the check cell below it. It prints *correct ✓* or *not yet ✗*.
7. If it prints *not yet*, send the repair prompt below to the assistant together with the code. If the result is still wrong, run the *Ready code* cell. It always gives the correct result, so the next step can start.

**"data" and "answer".** As in Week 1, the data cell of each step prepares the table "data" that the prompt describes. The assistant keeps the result under the name "answer", and the check cell reads it from there.

**At the start of every stop** run its preparation cell once. It loads the data of the stop and also runs Section 0 if this has not happened in the session, so a stop can also be started on another day or straight from the lecture page.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-07/PROMPTS.md), where GitHub shows a copy button next to each one.

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

## Stop 1: Transfer to echocardiography: A quality gate and the mini project

*Lecture: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#stop-1)* · *The same result on the Python path: Part 2, Sections 1 to 4*

A vision transformer pretrained on photographs, ViT-B/16, turns every echocardiography frame of CAMUS into 768 numbers, and a logistic regression learns from them which frames the experts rated as poor. The same gate is built from four simple image statistics and from the convolutional network of Week 4, EfficientNet-B0, and step 1.4 asks how sure the difference between the two networks is with 50 test patients. Step 1.5 is the mini project of the session: A mini vision transformer learns the photographs of CIFAR-10 and then a reduced ejection fraction from pairs of echocardiography frames of EchoNet-Dynamic or CAMUS.

**At the end of this stop:** The test AUC of the gate from four image statistics (0.704), from the class token of ViT-B/16 (0.862) and from the features of EfficientNet-B0 (0.782), the share of bootstrap samples of the test patients in which ViT-B/16 does better (0.925), and, in the mini project, the accuracy of the mini vision transformer on CIFAR-10 and the test AUC of its two starts on echocardiograms.

In [ ]:
#@title Preparation of Stop 1: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
if "echo" not in globals():
    camus_table = camus.table()
    echo, echo_patient, echo_view, echo_quality, echo_split = [], [], [], [], []
    for _, row in camus_table.iterrows():
        for view in ("2CH", "4CH"):
            img, _, _ = camus.load(row["patient"], view, "ED")
            echo.append(np.asarray(Image.fromarray(img).resize((224, 224), Image.BILINEAR)) / 255)
            echo_patient.append(row["patient"])
            echo_view.append(view)
            echo_quality.append(row["quality_" + view.lower()])
            echo_split.append(row["split"])
    echo = np.clip(np.stack(echo), 0, 1).astype(np.float32)
    echo_patient, echo_view, echo_quality, echo_split = map(np.array, (echo_patient, echo_view, echo_quality, echo_split))
    etr, eva, ete = echo_split == "training", echo_split == "validation", echo_split == "testing"
    poor = (echo_quality == "Poor").astype(int)
_SETS = {"training": "training", "validation": "validation", "testing": "test"}
_QUALITY = {"Good": "good", "Medium": "medium", "Poor": "poor"}


def _frames():
    """The 600 end-diastolic frames of CAMUS with their set, patient, view and image quality, as the prompts describe them."""
    return pd.DataFrame({"set": [_SETS[s] for s in echo_split], "patient": echo_patient, "view": echo_view,
                         "quality": [_QUALITY[q] for q in echo_quality], "frame": list(echo)})


def _course_gate():
    """The probabilities of poor quality that the gates of the course give to the 100 test frames, from the features of ViT-B/16 and of EfficientNet-B0 (computed once, about two minutes)."""
    if "_gate_cache" in globals():
        return _gate_cache
    vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).eval()
    vit.heads = nn.Identity()      # without its last layer, the network returns the class token: 768 numbers per image
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


    def to_imagenet(images, size=224):
        """Grey images with values from 0 to 1 → tensors of 3 x 224 x 224 with the mean and the standard deviation of ImageNet."""
        x = torch.as_tensor(np.asarray(images, dtype=np.float32))[:, None]
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        return (x - MEAN) / STD


    @torch.no_grad()
    def vit_features(images, batch=16):
        """The class token of ViT-B/16 for every image: an array of n x 768 numbers."""
        return torch.cat([vit(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


    @torch.no_grad()
    def vit_attention(image):
        """The attention of the 12 layers of ViT-B/16 for one image, as the mean of the 12 heads: an array of 12 x 197 x 197."""
        x = vit._process_input(to_imagenet([image]))
        x = torch.cat([vit.class_token, x], dim=1) + vit.encoder.pos_embedding
        maps = []
        for block in vit.encoder.layers:
            y = block.ln_1(x)
            out, w = block.self_attention(y, y, y, need_weights=True, average_attn_weights=True)
            maps.append(w[0].numpy())
            x = x + out
            x = x + block.mlp(block.ln_2(x))
        return np.stack(maps)


    def rollout(maps):
        """Attention rollout: the attention of each layer, with the residual connection added, multiplied from the first layer to the last."""
        r = np.eye(maps.shape[-1])
        for a in maps:
            a = 0.5 * a + 0.5 * np.eye(len(a))
            r = (a / a.sum(axis=1, keepdims=True)) @ r
        return r

    effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
    effnet.classifier = nn.Identity()      # without its last layer, the network returns the mean of its 1280 channels


    @torch.no_grad()
    def effnet_features(images, batch=16):
        """The 1280 features of EfficientNet-B0 for every image (Week 4)."""
        return torch.cat([effnet(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


    def image_statistics(images):
        """Four numbers per image: mean grey value, standard deviation, mean difference between neighbouring pixels and share of pixels brighter than half of the maximum."""
        return np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in images])


    def fit_gate(features, target):
        """A logistic regression on standardised features; the inverse regularisation strength is chosen on the validation frames."""
        best = None
        for C in (0.001, 0.01, 0.1, 1.0):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(features[etr], target[etr])
            a = roc_auc_score(target[eva], model.predict_proba(features[eva])[:, 1])
            if best is None or a > best[0]:
                best = (a, C, model)
        return best[2], best[1], best[0]


    def bootstrap_auc(target, prob, groups, n=2000, seed=2026):
        """AUC in 2000 bootstrap samples that draw groups (patients or images) with replacement."""
        rng = np.random.default_rng(seed)
        names = np.unique(groups)
        rows = {g: np.where(groups == g)[0] for g in names}
        out = []
        for _ in range(n):
            idx = np.concatenate([rows[g] for g in rng.choice(names, len(names))])
            if 0 < target[idx].sum() < len(idx):
                out.append(roc_auc_score(target[idx], prob[idx]))
        return np.array(out)
    out = {}
    for name, features in (("vit", vit_features(echo)), ("effnet", effnet_features(echo))):
        model, _, _ = fit_gate(features, poor)
        out[name] = model.predict_proba(features[ete])[:, 1]
    globals()["_gate_cache"] = out
    return out
print(len(echo), "frames of", len(set(echo_patient)), "patients")
print("Stop 1 is ready.")

### Step 1.1: A gate from four image statistics

**Goal.** Build a first quality gate from four numbers that describe the brightness, the contrast and the sharpness of a frame.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to flag echocardiography frames of poor quality before an automatic measurement of the ejection fraction, and starts with four simple numbers of each frame.  
> Data: The notebook holds a table named "data" with 600 apical echocardiography frames of CAMUS, a public data set: the end-diastolic frames of the two-chamber and of the four-chamber views of 300 patients. Each row is a frame: "set" gives the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients), "patient" the patient, "view" the view ("2CH" or "4CH") and "quality" the image quality given by the experts of CAMUS ("good", "medium" or "poor"). Each cell of the column "frame" holds the frame as an image of 224 × 224 pixels with grey values from 0 to 1.  
> Task: For every frame compute four numbers: the mean of its grey values, their standard deviation, the mean absolute difference between vertically neighbouring pixels, and the share of pixels brighter than half of the brightest pixel of the frame. The target is 1 for a frame of poor quality and 0 otherwise. Standardise the features with the training frames. Fit a logistic regression on the training frames with each of the inverse regularisation strengths 0.001, 0.01, 0.1 and 1, keep the one with the largest area under the ROC curve (AUC) on the validation frames, and compute the AUC of its probabilities on the test frames.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The AUC must lie between 0 and 1, and the test frames must not be used before the last step.  

In [ ]:
#@title Data of step 1.1: run this cell (it prepares the table "data" and shows it)
data = _frames()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.1
teach.check_step("Step 1.1", "answer", globals())

In [ ]:
#@title Ready code of step 1.1 (run it only if the code of the assistant does not pass the check)
data = _frames()
sets = data["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
target = (data["quality"] == "poor").astype(int).to_numpy()


def gate(features):
    best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(features[tr_], target[tr_])
        auc_val = roc_auc_score(target[va_], model.predict_proba(features[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    auc_test = roc_auc_score(target[te_], best[2].predict_proba(features[te_])[:, 1])
    print("inverse regularisation strength", best[1], "| validation AUC", round(best[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

frames = np.stack(data["frame"])
statistics = np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in frames])
answer = gate(statistics)
teach.check_step("Step 1.1", "answer", globals())

**What the result shows.** Four numbers give a test AUC of 0.704: Poor frames are often darker and less sharp, but these numbers know nothing of the heart.

### Step 1.2: The class token of a vision transformer

**Goal.** Describe every frame by the 768 numbers of the class token of ViT-B/16, pretrained on ImageNet, and build the gate on them.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a vision transformer that learned from millions of photographs, and never saw an echocardiogram, can describe frames well enough to flag those of poor quality.  
> Data: The notebook holds a table named "data" with 600 apical echocardiography frames of CAMUS, a public data set: the end-diastolic frames of the two-chamber and of the four-chamber views of 300 patients. Each row is a frame: "set" gives the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients), "patient" the patient, "view" the view ("2CH" or "4CH") and "quality" the image quality given by the experts of CAMUS ("good", "medium" or "poor"). Each cell of the column "frame" holds the frame as an image of 224 × 224 pixels with grey values from 0 to 1.  
> Task: Use the vision transformer ViT-B/16 with its ImageNet weights, without its last layer, so that it returns the class token of its last layer, 768 numbers for each frame. Give it every frame as an image of 224 × 224 pixels with the same grey values in its three colour channels, scaled with the mean and the standard deviation of ImageNet: Subtract 0.485, 0.456 and 0.406 from the three channels and divide them by 0.229, 0.224 and 0.225. Pass the frames through the network in groups of 16, in evaluation mode and without recording gradients. Then use the 768 numbers as the features of the gate. The target is 1 for a frame of poor quality and 0 otherwise. Standardise the features with the training frames. Fit a logistic regression on the training frames with each of the inverse regularisation strengths 0.001, 0.01, 0.1 and 1, keep the one with the largest area under the ROC curve (AUC) on the validation frames, and compute the AUC of its probabilities on the test frames.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: Every frame must give 768 numbers, and the AUC must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.2: run this cell (it prepares the table "data" and shows it)
data = _frames()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.2
teach.check_step("Step 1.2", "answer", globals())

In [ ]:
#@title Ready code of step 1.2 (run it only if the code of the assistant does not pass the check)
data = _frames()
sets = data["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
target = (data["quality"] == "poor").astype(int).to_numpy()


def gate(features):
    best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(features[tr_], target[tr_])
        auc_val = roc_auc_score(target[va_], model.predict_proba(features[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    auc_test = roc_auc_score(target[te_], best[2].predict_proba(features[te_])[:, 1])
    print("inverse regularisation strength", best[1], "| validation AUC", round(best[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

frames = np.stack(data["frame"]).astype(np.float32)


@torch.no_grad()
def features_of(net):
    net.eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(frames), 16):
        x = torch.as_tensor(frames[i:i + 16])[:, None].repeat(1, 3, 1, 1)
        out.append(net((x - mean) / std))
    return torch.cat(out).numpy()

vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
vit.heads = nn.Identity()
features = features_of(vit)
print(features.shape)
answer = gate(features)
teach.check_step("Step 1.2", "answer", globals())

**What the result shows.** The class token of ViT-B/16 gives a test AUC of 0.862. The network never saw an echocardiogram, but its 768 numbers describe edges, textures and shapes well enough for a gate.

### Step 1.3: The features of a convolutional network

**Goal.** Build the same gate on the 1,280 features of EfficientNet-B0, the network of Week 4.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare a vision transformer with the convolutional network used in Week 4 as the source of features for a quality gate on echocardiography frames.  
> Data: The notebook holds a table named "data" with 600 apical echocardiography frames of CAMUS, a public data set: the end-diastolic frames of the two-chamber and of the four-chamber views of 300 patients. Each row is a frame: "set" gives the set of the patient ("training", 200 patients; "validation", 50 patients; "test", 50 patients), "patient" the patient, "view" the view ("2CH" or "4CH") and "quality" the image quality given by the experts of CAMUS ("good", "medium" or "poor"). Each cell of the column "frame" holds the frame as an image of 224 × 224 pixels with grey values from 0 to 1.  
> Task: Use the convolutional network EfficientNet-B0 with its ImageNet weights, without its last layer, so that it returns the output of its average pooling, 1,280 numbers for each frame. Give it every frame as an image of 224 × 224 pixels with the same grey values in its three colour channels, scaled with the mean and the standard deviation of ImageNet: Subtract 0.485, 0.456 and 0.406 from the three channels and divide them by 0.229, 0.224 and 0.225. Pass the frames through the network in groups of 16, in evaluation mode and without recording gradients. Then use the 1,280 numbers as the features of the gate. The target is 1 for a frame of poor quality and 0 otherwise. Standardise the features with the training frames. Fit a logistic regression on the training frames with each of the inverse regularisation strengths 0.001, 0.01, 0.1 and 1, keep the one with the largest area under the ROC curve (AUC) on the validation frames, and compute the AUC of its probabilities on the test frames.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen strength and the validation and test AUC, and keep the test AUC, with three decimals, under the name "answer".  
> Test: Every frame must give 1,280 numbers, and the AUC must lie between 0 and 1.  

In [ ]:
#@title Data of step 1.3: run this cell (it prepares the table "data" and shows it)
data = _frames()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.3
teach.check_step("Step 1.3", "answer", globals())

In [ ]:
#@title Ready code of step 1.3 (run it only if the code of the assistant does not pass the check)
data = _frames()
sets = data["set"].to_numpy()
tr_, va_, te_ = sets == "training", sets == "validation", sets == "test"
target = (data["quality"] == "poor").astype(int).to_numpy()


def gate(features):
    best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(features[tr_], target[tr_])
        auc_val = roc_auc_score(target[va_], model.predict_proba(features[va_])[:, 1])
        if best is None or auc_val > best[0]:
            best = (auc_val, C, model)
    auc_test = roc_auc_score(target[te_], best[2].predict_proba(features[te_])[:, 1])
    print("inverse regularisation strength", best[1], "| validation AUC", round(best[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

frames = np.stack(data["frame"]).astype(np.float32)


@torch.no_grad()
def features_of(net):
    net.eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(frames), 16):
        x = torch.as_tensor(frames[i:i + 16])[:, None].repeat(1, 3, 1, 1)
        out.append(net((x - mean) / std))
    return torch.cat(out).numpy()

effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
effnet.classifier = nn.Identity()
features = features_of(effnet)
print(features.shape)
answer = gate(features)
teach.check_step("Step 1.3", "answer", globals())

**What the result shows.** EfficientNet-B0 gives a test AUC of 0.782, against 0.862 for ViT-B/16. Both networks learned from the same photographs; the difference is small next to the 15 poor test frames on which it rests.

### Step 1.4: How sure is the difference?

**Goal.** Draw the 50 test patients again and again with replacement and see how often the vision transformer wins.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether the difference between two quality gates for echocardiography, measured on the frames of 50 test patients, would hold on other patients.  
> Data: The notebook holds a table named "data" with the 100 test frames of CAMUS, a public data set of apical echocardiograms: the end-diastolic frames of the two-chamber and of the four-chamber views of 50 patients. Each row is a frame: "patient" gives the patient, "view" the view ("2CH" or "4CH"), "poor" is 1 when the experts of CAMUS rated the image quality as poor and 0 otherwise, and the two columns "probability, ViT-B/16" and "probability, EfficientNet-B0" give the probability of poor quality from two quality gates of the course, logistic regressions on the features of two networks pretrained on ImageNet.  
> Task: Compute the AUC of both columns of probabilities against the column "poor". Then draw 2,000 bootstrap samples: Each sample draws 50 patients from the 50 test patients with replacement and keeps both frames of every patient drawn. Fix the random seed to 2026 so that the result can be repeated. In every sample that contains frames of both kinds, compute the AUC of both gates and the difference, ViT-B/16 minus EfficientNet-B0. Report the mean difference, its 95 percent interval from the 2.5th to the 97.5th percentile, and the share of samples in which the difference is above 0.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two AUCs, the mean difference and its interval, and keep the share, with three decimals, under the name "answer".  
> Test: The share must lie between 0 and 1, and the two frames of a patient must always be drawn together.  

In [ ]:
#@title Data of step 1.4: run this cell (it prepares the table "data" and shows it)
_g = _course_gate()   # about two minutes the first time: the features of the 600 frames are computed once
_t = _frames().loc[ete].reset_index(drop=True)
data = pd.DataFrame({"patient": _t["patient"], "view": _t["view"], "poor": (_t["quality"] == "poor").astype(int),
                     "probability, ViT-B/16": _g["vit"], "probability, EfficientNet-B0": _g["effnet"]})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.4
teach.check_step("Step 1.4", "answer", globals())

In [ ]:
#@title Ready code of step 1.4 (run it only if the code of the assistant does not pass the check)
_g = _course_gate()   # about two minutes the first time: the features of the 600 frames are computed once
_t = _frames().loc[ete].reset_index(drop=True)
data = pd.DataFrame({"patient": _t["patient"], "view": _t["view"], "poor": (_t["quality"] == "poor").astype(int),
                     "probability, ViT-B/16": _g["vit"], "probability, EfficientNet-B0": _g["effnet"]})
y = data["poor"].to_numpy()
pv = data["probability, ViT-B/16"].to_numpy()
pe = data["probability, EfficientNet-B0"].to_numpy()
print("AUC ViT-B/16", round(roc_auc_score(y, pv), 3), "| AUC EfficientNet-B0", round(roc_auc_score(y, pe), 3))
patients = data["patient"].to_numpy()
names = np.unique(patients)
rows = {p: np.where(patients == p)[0] for p in names}
rng = np.random.default_rng(2026)
differences = []
for _ in range(2000):
    idx = np.concatenate([rows[p] for p in rng.choice(names, len(names))])
    if 0 < y[idx].sum() < len(idx):
        differences.append(roc_auc_score(y[idx], pv[idx]) - roc_auc_score(y[idx], pe[idx]))
differences = np.array(differences)
print("mean difference", round(differences.mean(), 3), "| 95 percent interval", np.round(np.percentile(differences, [2.5, 97.5]), 3))
answer = round(float((differences > 0).mean()), 3)
teach.check_step("Step 1.4", "answer", globals())

**What the result shows.** ViT-B/16 does better in a share of 0.925 of the samples. The interval of the difference contains 0: With 50 test patients and 15 poor frames, the data do not decide between the two networks.

### Step 1.5: Mini project: A mini vision transformer from CIFAR-10 to echocardiography

**Goal.** Train a small vision transformer on the photographs of CIFAR-10, then fine-tune it on pairs of echocardiography frames to recognise a reduced ejection fraction, and compare it with the same network trained on the echocardiograms alone.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a small vision transformer that first learned from everyday photographs recognises a reduced ejection fraction on echocardiograms better than the same network that learned from the echocardiograms alone.  
> Data: The notebook holds a table named "data" with two kinds of rows. The rows whose "source" is "CIFAR-10" are the 60,000 colour photographs of CIFAR-10, a public data set of ten everyday classes such as airplanes, cats and ships: "set" gives "training" (50,000 photographs) or "test" (10,000), each cell of "image" holds a photograph as an array of 32 × 32 × 3 numbers from 0 to 255, and "class" gives its class as a number from 0 to 9. The rows whose "source" is "echo" are pairs of apical echocardiography frames of EchoNet-Dynamic or of CAMUS, two public data sets: "set" gives "training", "validation" or "test", "patient" the patient or the video, each cell of "image" holds an array of 32 × 32 × 3 numbers whose three channels are the end-diastolic frame and the end-systolic frame, with values from 0 to 1, and their difference, with values from −1 to 1, and "reduced EF" is 1 when the ejection fraction is below 50 percent and 0 otherwise.  
> Task: Build a small vision transformer for colour images of 32 × 32 pixels. Cut the image into 64 squares of 4 × 4 pixels and map every square to 64 numbers with the same learned linear map. Put a learned class token of 64 numbers before the 64 tokens and add learned positions, one vector of 64 numbers for each of the 65 places, started from small random numbers with a standard deviation of 0.02. Pass the tokens through four transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with ten outputs, the scores of the ten classes. Train it on the CIFAR-10 rows of the training set: Divide the photographs by 255 and standardise each colour channel with the mean and the standard deviation of the training photographs. Train for 5 epochs in shuffled groups of 128 photographs with the cross-entropy loss and the AdamW optimiser with a learning rate of 0.001 and a weight decay of 0.05. Move every group by a random shift of up to 4 pixels in each direction, with the borders filled by reflection, and mirror it from left to right with a probability of one half. Fix the random seed to 2026 so that the result can be repeated. Compute the accuracy on the 10,000 test photographs. Then make two networks for the echocardiography rows, each ending in a linear layer with one output, the logit of a reduced ejection fraction: one with the same structure and new random weights, and one as a copy of the network trained on CIFAR-10 whose last layer is replaced by a new layer with one output. Use the echocardiography images as they are. Train each network for 30 epochs on the training rows in shuffled groups of 32 with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.0005. After every epoch, compute the AUC on the validation rows, and keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Compute the test AUC of both networks against the column "reduced EF".  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the accuracy on CIFAR-10 and the test AUC of both networks, and keep the test AUC of the network that started from CIFAR-10, with three decimals, under the name "answer".  
> Test: The accuracy on CIFAR-10 must lie well above 0.1, which one class in ten gives by chance, and the test rows must not be used before the end.  

In [ ]:
#@title Data of step 1.5: run this cell (it prepares the table "data" and shows it)
class Attention(nn.Module):
    """Self-attention with several heads; the weights of the last call are kept for the maps."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """The number of trainable weights of a network."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]


def cifar10():
    """CIFAR-10 from its source at the University of Toronto, about 170 MB, downloaded once into data/cifar10: 50,000 training and 10,000 test images of 32 x 32 pixels with three colour channels, and their classes."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """Images of n x 32 x 32 x 3 (values 0 to 255) → tensors of n x 3 x 32 x 32, every channel standardised with the mean and the standard deviation of the training images."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Data augmentation: the group is moved by up to 4 pixels in each direction and mirrored from left to right half of the time."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """AdamW with a learning rate of 0.001 and a weight decay of 0.05 over shuffled groups of 128 images, with the cross-entropy of the ten classes."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: training loss {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """The share of images whose class gets the highest score."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """The end-diastolic and end-systolic frames of both views of the 300 patients of CAMUS as pairs of 32 x 32 pixels, with the EF of the patient."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"set": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "patient": r["patient"],
                         "pair": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """The two traced frames of EchoNet-Dynamic videos as pairs of 32 x 32 pixels, with the EF: 2,000 training, 400 validation and 400 test videos."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"set": split, "patient": d["file"], "pair": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """A pair of frames → three channels: the end-diastolic frame, the end-systolic frame and their difference, in the places of the red, green and blue of a photograph."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Adam over shuffled groups of 32 pairs with the binary cross-entropy of a reduced EF; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    _cifar = cifar10()
except Exception as _error:
    _cifar = None
    print("CIFAR-10 could not be downloaded:", type(_error).__name__)
try:
    _source = echonet.find()
except FileNotFoundError:
    _source = None
_pairs = echonet_pairs(_source) if _source is not None else camus_pairs()
_SET = {"TRAIN": "training", "VAL": "validation", "TEST": "test"}
_rows = []
if _cifar is not None:
    for _part, (_images, _labels) in _cifar.items():
        _rows.append(pd.DataFrame({"source": "CIFAR-10", "set": "training" if _part == "training" else "test",
                                   "patient": None, "image": list(_images), "class": _labels, "reduced EF": np.nan}))
_rows.append(pd.DataFrame({"source": "echo", "set": _pairs["set"].map(_SET).to_numpy(),
                           "patient": _pairs["patient"].to_numpy(),
                           "image": list(three_channels(_pairs["pair"]).transpose(0, 2, 3, 1)),
                           "class": np.nan, "reduced EF": (_pairs["EF"] < 50).astype(int).to_numpy()}))
data = pd.concat(_rows, ignore_index=True)
print("echocardiograms of", "EchoNet-Dynamic" if _source is not None else "CAMUS")
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 1.5
if globals().get("_cifar") is None:
    print("CIFAR-10 could not be downloaded here, so this step cannot run on this computer. Run it in Google Colab or on a computer with internet access.")
else:
    teach.check_range("Step 1.5", "answer", 0.5, 1.0, globals())

In [ ]:
#@title Ready code of step 1.5 (run it only if the code of the assistant does not pass the check)
class Attention(nn.Module):
    """Self-attention with several heads; the weights of the last call are kept for the maps."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """The number of trainable weights of a network."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]


def cifar10():
    """CIFAR-10 from its source at the University of Toronto, about 170 MB, downloaded once into data/cifar10: 50,000 training and 10,000 test images of 32 x 32 pixels with three colour channels, and their classes."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """Images of n x 32 x 32 x 3 (values 0 to 255) → tensors of n x 3 x 32 x 32, every channel standardised with the mean and the standard deviation of the training images."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Data augmentation: the group is moved by up to 4 pixels in each direction and mirrored from left to right half of the time."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """AdamW with a learning rate of 0.001 and a weight decay of 0.05 over shuffled groups of 128 images, with the cross-entropy of the ten classes."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: training loss {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """The share of images whose class gets the highest score."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """The end-diastolic and end-systolic frames of both views of the 300 patients of CAMUS as pairs of 32 x 32 pixels, with the EF of the patient."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"set": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "patient": r["patient"],
                         "pair": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """The two traced frames of EchoNet-Dynamic videos as pairs of 32 x 32 pixels, with the EF: 2,000 training, 400 validation and 400 test videos."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"set": split, "patient": d["file"], "pair": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """A pair of frames → three channels: the end-diastolic frame, the end-systolic frame and their difference, in the places of the red, green and blue of a photograph."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Adam over shuffled groups of 32 pairs with the binary cross-entropy of a reduced EF; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    _cifar = cifar10()
except Exception as _error:
    _cifar = None
    print("CIFAR-10 could not be downloaded:", type(_error).__name__)
try:
    _source = echonet.find()
except FileNotFoundError:
    _source = None
_pairs = echonet_pairs(_source) if _source is not None else camus_pairs()
_SET = {"TRAIN": "training", "VAL": "validation", "TEST": "test"}
_rows = []
if _cifar is not None:
    for _part, (_images, _labels) in _cifar.items():
        _rows.append(pd.DataFrame({"source": "CIFAR-10", "set": "training" if _part == "training" else "test",
                                   "patient": None, "image": list(_images), "class": _labels, "reduced EF": np.nan}))
_rows.append(pd.DataFrame({"source": "echo", "set": _pairs["set"].map(_SET).to_numpy(),
                           "patient": _pairs["patient"].to_numpy(),
                           "image": list(three_channels(_pairs["pair"]).transpose(0, 2, 3, 1)),
                           "class": np.nan, "reduced EF": (_pairs["EF"] < 50).astype(int).to_numpy()}))
data = pd.concat(_rows, ignore_index=True)
print("echocardiograms of", "EchoNet-Dynamic" if _source is not None else "CAMUS")
if globals().get("_cifar") is None:
    print("CIFAR-10 could not be downloaded here, so this step cannot run on this computer. Run it in Google Colab or on a computer with internet access.")
else:
    _photos = data[data["source"] == "CIFAR-10"]
    _train, _test = _photos[_photos["set"] == "training"], _photos[_photos["set"] == "test"]
    _images = np.stack(_train["image"])
    _mean, _std = _images.mean(axis=(0, 1, 2)) / 255, _images.std(axis=(0, 1, 2)) / 255
    torch.manual_seed(2026)
    photo_net = SmallViT(size=32, patch=4, channels=3, outputs=10)
    train_cifar(photo_net, to_tensor(_images, _mean, _std), torch.as_tensor(_train["class"].to_numpy(int)), epochs=5)
    _accuracy = accuracy(photo_net, to_tensor(np.stack(_test["image"]), _mean, _std), torch.as_tensor(_test["class"].to_numpy(int)))
    print("accuracy on CIFAR-10:", round(_accuracy, 3))
    _echo = data[data["source"] == "echo"]
    _X = {s: torch.as_tensor(np.stack(_echo[_echo["set"] == s]["image"]).transpose(0, 3, 1, 2).copy()) for s in ("training", "validation", "test")}
    _Y = {s: torch.as_tensor(_echo[_echo["set"] == s]["reduced EF"].to_numpy(np.float32)) for s in _X}
    _aucs = {}
    for _start in ("random weights", "CIFAR-10"):
        torch.manual_seed(2026)
        if _start == "CIFAR-10":
            _net = copy.deepcopy(photo_net)
            _net.head = nn.Linear(64, 1)
        else:
            _net = SmallViT(size=32, patch=4, channels=3)
        fine_tune(_net, _X["training"], _Y["training"], _X["validation"], _Y["validation"])
        with torch.no_grad():
            _aucs[_start] = round(float(roc_auc_score(_Y["test"].numpy(), torch.sigmoid(_net(_X["test"])).numpy())), 3)
    print("test AUC:", _aucs)
    answer = _aucs["CIFAR-10"]
    teach.check_range("Step 1.5", "answer", 0.5, 1.0, globals())

**What the result shows.** The accuracy on CIFAR-10 shows what the small network learned from photographs in five epochs; one class in ten would be right by chance. The two test AUCs show whether this start helps on echocardiograms. Both rest on few test patients, 50 of CAMUS or 400 videos of EchoNet-Dynamic, and change from one computer to the next, so the check accepts an AUC from 0.5 to 1.0 for the start from CIFAR-10.

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#last-two-hours-hybrid-cnn-vit-and-medical-imaging) and continue reading at «Last two hours: Hybrid CNN-ViT and medical imaging». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: Stenoses on synthetic and on real angiograms

*Lecture: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#stop-2)* · *The same result on the Python path: Part 2, Sections 5 to 6*

A small convolutional network learns to recognise patches of 64 × 64 pixels centred on a significant stenosis, first on the synthetic angiograms of the virtual clinic. The same network is then tested on the real angiograms of ARCADE, without any change, and finally trained again on the training angiograms of ARCADE.

**At the end of this stop:** The test AUC of the network on synthetic patches (0.965), the AUC of the same kind of network on the real patches of ARCADE when it learned only from synthetic ones (0.538), and its test AUC when it learns from ARCADE (0.81).

In [ ]:
#@title Preparation of Stop 2: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_SETS = {"training": "training", "validation": "validation", "test": "test", "train": "training", "val": "validation"}


def _synthetic_table():
    """Patches of the synthetic angiograms of 300 training, 100 validation and 100 test patients of the virtual clinic."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"set": _SETS[s], "patient": ids, "patch": list(X), "stenosis": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """Patches of the 1500 angiograms of ARCADE (stenosis set) with their set and angiogram."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"set": _SETS[s], "angiogram": ids, "patch": list(X), "stenosis": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """A network of the course, trained once on the training patches and kept for the next steps (one to four minutes)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Every patch on its own: mean grey value 0 and standard deviation 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """Convolution of 3 x 3, batch normalisation, ReLU and max pooling of 2 x 2: half the size, b channels."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Four convolution blocks (16, 32, 64 and 64 channels), the mean of every channel and one output."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Self-attention with several heads; the weights of the last call are kept for the maps."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """The number of trainable weights of a network."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """A hybrid: three convolution blocks turn the patch into 8 x 8 positions of 64 numbers, the tokens of two transformer blocks."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """The probability of a stenosis for every patch."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001, groups of 64 patches turned by a random quarter turn and mirrored half of the time; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["set"].to_numpy()
    X, y = standardise(np.stack(t["patch"])), t["stenosis"].to_numpy()
    tr_, va_ = sets == "training", sets == "validation"
    torch.manual_seed(2026)
    model = SmallCNN() if kind == "synthetic_cnn" else SmallViT()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]

print("The tables of Stop 2 are made when their steps run: the synthetic patches in about two minutes, ARCADE (451.6 MB, once) and its patches in a few minutes.")
print("Stop 2 is ready.")

### Step 2.1: A small convolutional network on synthetic angiograms

**Goal.** Train a small convolutional network on patches of the synthetic angiograms and measure it on the patches of the test patients.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a small network can learn from coronary angiograms whether a small square of an image is centred on a significant stenosis.  
> Data: The notebook holds a table named "data" with patches of 64 × 64 pixels cut from the synthetic coronary angiograms of the virtual clinic, two views for each of 500 patients (training 300 patients, validation 100, test 100). Each row is a patch: "set" gives its set ("training", "validation" or "test"), "patient" the patient, and each cell of "patch" holds the patch as an image of 64 × 64 pixels with grey values from 0 to 1. "stenosis" is 1 when the patch is centred within 6 pixels of a significant stenosis (50 percent or more in the left main coronary artery, 70 percent or more elsewhere) and 0 when it lies on a vessel at least 48 pixels from every stenosis.  
> Task: Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build a small convolutional network for one patch of 64 × 64 pixels with one channel: four blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32, 64 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2; then the mean of every channel over its positions and a linear layer with one output, the logit of a stenosis. Train the network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then compute the probabilities of the test patches and their AUC against the column "stenosis".  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable weights and the validation AUC of every epoch, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The network must have 60,641 trainable weights, and the test patches must not be used before the last step.  

In [ ]:
#@title Data of step 2.1: run this cell (it prepares the table "data" and shows it)
data = _synthetic_table()   # about two minutes: the virtual clinic draws 1000 angiograms
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.1
teach.check_step("Step 2.1", "answer", globals())

In [ ]:
#@title Ready code of step 2.1 (run it only if the code of the assistant does not pass the check)
data = _synthetic_table()   # about two minutes: the virtual clinic draws 1000 angiograms


def standardise(patches):
    patches = np.asarray(patches, dtype=np.float32)
    m = patches.mean(axis=(1, 2), keepdims=True)
    s = patches.std(axis=(1, 2), keepdims=True)
    return (patches - m) / (s + 0.02)


def block(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 64))
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)

torch.manual_seed(2026)
net = Net()
print("trainable weights:", sum(p.numel() for p in net.parameters() if p.requires_grad))
sets = data["set"].to_numpy()
X = torch.as_tensor(standardise(np.stack(data["patch"])))[:, None]
y = torch.as_tensor(data["stenosis"].to_numpy(), dtype=torch.float32)
tr_, va_, te_ = [torch.as_tensor(sets == s) for s in ("training", "validation", "test")]


@torch.no_grad()
def probabilities(X_):
    net.eval()
    return torch.cat([torch.sigmoid(net(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xtr, ytr = X[tr_], y[tr_]
torch.manual_seed(2026)
optimiser = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
best, state = -1.0, None
for epoch in range(8):
    net.train()
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = order[i:i + 64]
        xb = torch.rot90(Xtr[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xb = torch.flip(xb, dims=(3,))
        loss = loss_fn(net(xb), ytr[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    auc = roc_auc_score(y[va_].numpy(), probabilities(X[va_]))
    print(f"epoch {epoch + 1}: validation AUC {auc:.3f}")
    if auc > best:
        best, state = auc, copy.deepcopy(net.state_dict())
net.load_state_dict(state)
answer = round(float(roc_auc_score(y[te_].numpy(), probabilities(X[te_]))), 3)
print("test AUC:", answer)
teach.check_step("Step 2.1", "answer", globals())

**What the result shows.** On synthetic patches the network reaches a test AUC of 0.965. The task looks solved, but only for the images that the virtual clinic draws.

### Step 2.2: The same network on real angiograms

**Goal.** Test a network that learned only from synthetic angiograms on the real angiograms of ARCADE.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how a network that learned to find stenoses on synthetic angiograms does on real X-ray angiograms that it has never seen.  
> Data: The notebook holds a table named "data" with the test patches of two sources: 1,700 patches of 64 × 64 pixels from the synthetic coronary angiograms of the virtual clinic and 3,088 from the real X-ray angiograms of ARCADE, a public data set. Each row is a patch: "source" gives its source ("synthetic" or "ARCADE"), "stenosis" is 1 for a patch centred on a stenosis and 0 for a patch on a vessel far from every stenosis, and "probability" is the probability of a stenosis given by a small convolutional network of the course that learned from the synthetic training patches only.  
> Task: Compute, for each source separately, the area under the ROC curve (AUC) of the column "probability" against the column "stenosis", and the difference, synthetic minus ARCADE.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two AUCs and their difference, and keep the AUC of the source "ARCADE", with three decimals, under the name "answer".  
> Test: Each AUC must be computed from the patches of its own source only.  

In [ ]:
#@title Data of step 2.2: run this cell (it prepares the table "data" and shows it)
_m, _std, _pred = _course_model("synthetic_cnn")   # a few minutes the first time: the network of the course learns the synthetic patches once
_s = _synthetic_table()
_s = _s[_s["set"] == "test"]
_a = _arcade_table()
_a = _a[_a["set"] == "test"]
data = pd.DataFrame({"source": ["synthetic"] * len(_s) + ["ARCADE"] * len(_a),
                     "stenosis": np.concatenate([_s["stenosis"], _a["stenosis"]]),
                     "probability": np.concatenate([_pred(_m, _std(np.stack(_s["patch"]))), _pred(_m, _std(np.stack(_a["patch"])))])})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.2
teach.check_step("Step 2.2", "answer", globals())

In [ ]:
#@title Ready code of step 2.2 (run it only if the code of the assistant does not pass the check)
_m, _std, _pred = _course_model("synthetic_cnn")   # a few minutes the first time: the network of the course learns the synthetic patches once
_s = _synthetic_table()
_s = _s[_s["set"] == "test"]
_a = _arcade_table()
_a = _a[_a["set"] == "test"]
data = pd.DataFrame({"source": ["synthetic"] * len(_s) + ["ARCADE"] * len(_a),
                     "stenosis": np.concatenate([_s["stenosis"], _a["stenosis"]]),
                     "probability": np.concatenate([_pred(_m, _std(np.stack(_s["patch"]))), _pred(_m, _std(np.stack(_a["patch"])))])})
aucs = {}
for source, part in data.groupby("source"):
    aucs[source] = roc_auc_score(part["stenosis"], part["probability"])
print({k: round(v, 3) for k, v in aucs.items()}, "| difference", round(aucs["synthetic"] - aucs["ARCADE"], 3))
answer = round(float(aucs["ARCADE"]), 3)
teach.check_step("Step 2.2", "answer", globals())

**What the result shows.** On ARCADE the AUC falls to 0.538. Real angiograms have other textures, noise, overlapping structures and stenoses of every degree: The network learned the world of the virtual clinic, not the stenosis.

### Step 2.3: Training on real angiograms

**Goal.** Train the same small network on the patches of the training angiograms of ARCADE.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well a small convolutional network finds stenoses on real coronary angiograms when it learns from real angiograms.  
> Data: The notebook holds a table named "data" with patches of 64 × 64 pixels cut from the 1,500 X-ray coronary angiograms of the stenosis set of ARCADE, a public data set (training 1,000 angiograms, validation 200, test 300). Each row is a patch: "set" gives its set ("training", "validation" or "test"), "angiogram" the number of its angiogram, and each cell of "patch" holds the patch as an image of 64 × 64 pixels with grey values from 0 to 1. "stenosis" is 1 when the patch is centred within 6 pixels of the centre of a stenosis outlined by the experts and 0 when it lies on a vessel at least 48 pixels from every outlined stenosis and from the border of the image.  
> Task: Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build a small convolutional network for one patch of 64 × 64 pixels with one channel: four blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32, 64 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2; then the mean of every channel over its positions and a linear layer with one output, the logit of a stenosis. Train the network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then compute the probabilities of the test patches and their AUC against the column "stenosis".  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable weights and the validation AUC of every epoch, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The network must have 60,641 trainable weights, and no patch of a test angiogram may be used for training.  

In [ ]:
#@title Data of step 2.3: run this cell (it prepares the table "data" and shows it)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 2.3
teach.check_step("Step 2.3", "answer", globals())

In [ ]:
#@title Ready code of step 2.3 (run it only if the code of the assistant does not pass the check)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut


def standardise(patches):
    patches = np.asarray(patches, dtype=np.float32)
    m = patches.mean(axis=(1, 2), keepdims=True)
    s = patches.std(axis=(1, 2), keepdims=True)
    return (patches - m) / (s + 0.02)


def block(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 64))
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)

torch.manual_seed(2026)
net = Net()
print("trainable weights:", sum(p.numel() for p in net.parameters() if p.requires_grad))
sets = data["set"].to_numpy()
X = torch.as_tensor(standardise(np.stack(data["patch"])))[:, None]
y = torch.as_tensor(data["stenosis"].to_numpy(), dtype=torch.float32)
tr_, va_, te_ = [torch.as_tensor(sets == s) for s in ("training", "validation", "test")]


@torch.no_grad()
def probabilities(X_):
    net.eval()
    return torch.cat([torch.sigmoid(net(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xtr, ytr = X[tr_], y[tr_]
torch.manual_seed(2026)
optimiser = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
best, state = -1.0, None
for epoch in range(8):
    net.train()
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = order[i:i + 64]
        xb = torch.rot90(Xtr[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xb = torch.flip(xb, dims=(3,))
        loss = loss_fn(net(xb), ytr[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    auc = roc_auc_score(y[va_].numpy(), probabilities(X[va_]))
    print(f"epoch {epoch + 1}: validation AUC {auc:.3f}")
    if auc > best:
        best, state = auc, copy.deepcopy(net.state_dict())
net.load_state_dict(state)
answer = round(float(roc_auc_score(y[te_].numpy(), probabilities(X[te_]))), 3)
print("test AUC:", answer)
teach.check_step("Step 2.3", "answer", globals())

**What the result shows.** Trained on ARCADE, the network reaches a test AUC of 0.81, against 0.538 when it learned from synthetic angiograms. A model is as good as the images that it learned from.

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared) and continue reading at «A convolutional network, a vision transformer and a hybrid compared». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: A convolutional network, a vision transformer and a hybrid

*Lecture: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#stop-3)* · *The same result on the Python path: Part 2, Sections 7 to 9*

A small vision transformer and a hybrid of convolutions and attention learn the patches of ARCADE in the same way as the convolutional network of Stop 2, so that the three can be compared on the same test angiograms. The last step asks where the vision transformer looks: how much of the attention of its class token falls on the centre of the patch, where the stenosis is.

**At the end of this stop:** The test AUC of the small vision transformer (0.93) and of the hybrid (0.934), and the mean share of the attention of the class token on the central 16 × 16 pixels of the patches with a stenosis (0.153).

In [ ]:
#@title Preparation of Stop 3: run this cell once (it loads the data of the stop; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_SETS = {"training": "training", "validation": "validation", "test": "test", "train": "training", "val": "validation"}


def _synthetic_table():
    """Patches of the synthetic angiograms of 300 training, 100 validation and 100 test patients of the virtual clinic."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"set": _SETS[s], "patient": ids, "patch": list(X), "stenosis": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """Patches of the 1500 angiograms of ARCADE (stenosis set) with their set and angiogram."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"set": _SETS[s], "angiogram": ids, "patch": list(X), "stenosis": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """A network of the course, trained once on the training patches and kept for the next steps (one to four minutes)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Every patch on its own: mean grey value 0 and standard deviation 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """Convolution of 3 x 3, batch normalisation, ReLU and max pooling of 2 x 2: half the size, b channels."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Four convolution blocks (16, 32, 64 and 64 channels), the mean of every channel and one output."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Self-attention with several heads; the weights of the last call are kept for the maps."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """The number of trainable weights of a network."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """A hybrid: three convolution blocks turn the patch into 8 x 8 positions of 64 numbers, the tokens of two transformer blocks."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """The probability of a stenosis for every patch."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001, groups of 64 patches turned by a random quarter turn and mirrored half of the time; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["set"].to_numpy()
    X, y = standardise(np.stack(t["patch"])), t["stenosis"].to_numpy()
    tr_, va_ = sets == "training", sets == "validation"
    torch.manual_seed(2026)
    model = SmallCNN() if kind == "synthetic_cnn" else SmallViT()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]

_arcade_table()
print(len(_arcade_cache), "patches of ARCADE")
print("Stop 3 is ready.")

### Step 3.1: A small vision transformer

**Goal.** Train a small vision transformer, which cuts every patch into 64 squares and lets them attend to each other, on the patches of ARCADE.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare a small vision transformer with a small convolutional network for finding stenoses on real coronary angiograms, both trained in the same way.  
> Data: The notebook holds a table named "data" with patches of 64 × 64 pixels cut from the 1,500 X-ray coronary angiograms of the stenosis set of ARCADE, a public data set (training 1,000 angiograms, validation 200, test 300). Each row is a patch: "set" gives its set ("training", "validation" or "test"), "angiogram" the number of its angiogram, and each cell of "patch" holds the patch as an image of 64 × 64 pixels with grey values from 0 to 1. "stenosis" is 1 when the patch is centred within 6 pixels of the centre of a stenosis outlined by the experts and 0 when it lies on a vessel at least 48 pixels from every outlined stenosis and from the border of the image.  
> Task: Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build a small vision transformer for one patch of 64 × 64 pixels with one channel. Cut the patch into 64 squares of 8 × 8 pixels and map every square to 64 numbers with the same learned linear map. Put a learned class token of 64 numbers before the 64 tokens and add learned positions, one vector of 64 numbers for each of the 65 places, started from small random numbers with a standard deviation of 0.02. Pass the tokens through four transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with one output, the logit of a stenosis. Train the network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then compute the probabilities of the test patches and their AUC against the column "stenosis".  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable weights and the validation AUC of every epoch, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The network must have 142,465 trainable weights, and no patch of a test angiogram may be used for training.  

In [ ]:
#@title Data of step 3.1: run this cell (it prepares the table "data" and shows it)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.1
teach.check_step("Step 3.1", "answer", globals())

In [ ]:
#@title Ready code of step 3.1 (run it only if the code of the assistant does not pass the check)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut


def standardise(patches):
    patches = np.asarray(patches, dtype=np.float32)
    m = patches.mean(axis=(1, 2), keepdims=True)
    s = patches.std(axis=(1, 2), keepdims=True)
    return (patches - m) / (s + 0.02)


class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        w = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        return self.out((w @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    def __init__(self, dim=64, heads=4, hidden=128):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class Transformer(nn.Module):
    def __init__(self, tokens, depth, dim=64):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, tokens + 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[Block(dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 1)

    def forward(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        return self.head(self.norm(self.blocks(t))[:, 0]).squeeze(1)



class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Conv2d(1, 64, 8, stride=8)
        self.transformer = Transformer(64, 4)

    def forward(self, x):
        return self.transformer(self.embed(x).flatten(2).transpose(1, 2))

torch.manual_seed(2026)
net = Net()
print("trainable weights:", sum(p.numel() for p in net.parameters() if p.requires_grad))
sets = data["set"].to_numpy()
X = torch.as_tensor(standardise(np.stack(data["patch"])))[:, None]
y = torch.as_tensor(data["stenosis"].to_numpy(), dtype=torch.float32)
tr_, va_, te_ = [torch.as_tensor(sets == s) for s in ("training", "validation", "test")]


@torch.no_grad()
def probabilities(X_):
    net.eval()
    return torch.cat([torch.sigmoid(net(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xtr, ytr = X[tr_], y[tr_]
torch.manual_seed(2026)
optimiser = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
best, state = -1.0, None
for epoch in range(8):
    net.train()
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = order[i:i + 64]
        xb = torch.rot90(Xtr[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xb = torch.flip(xb, dims=(3,))
        loss = loss_fn(net(xb), ytr[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    auc = roc_auc_score(y[va_].numpy(), probabilities(X[va_]))
    print(f"epoch {epoch + 1}: validation AUC {auc:.3f}")
    if auc > best:
        best, state = auc, copy.deepcopy(net.state_dict())
net.load_state_dict(state)
answer = round(float(roc_auc_score(y[te_].numpy(), probabilities(X[te_]))), 3)
print("test AUC:", answer)
teach.check_step("Step 3.1", "answer", globals())

**What the result shows.** The small vision transformer reaches a test AUC of 0.93, against 0.81 for the convolutional network. With about 13,000 training patches, attention learns what the convolutions are given by their design.

### Step 3.2: A hybrid of convolutions and attention

**Goal.** Train a hybrid model: three convolution blocks first, then two transformer blocks on the positions of their output.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether a network that first looks at small neighbourhoods with convolutions and then relates the parts of the patch with attention finds stenoses on real angiograms better than either kind alone.  
> Data: The notebook holds a table named "data" with patches of 64 × 64 pixels cut from the 1,500 X-ray coronary angiograms of the stenosis set of ARCADE, a public data set (training 1,000 angiograms, validation 200, test 300). Each row is a patch: "set" gives its set ("training", "validation" or "test"), "angiogram" the number of its angiogram, and each cell of "patch" holds the patch as an image of 64 × 64 pixels with grey values from 0 to 1. "stenosis" is 1 when the patch is centred within 6 pixels of the centre of a stenosis outlined by the experts and 0 when it lies on a vessel at least 48 pixels from every outlined stenosis and from the border of the image.  
> Task: Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build a hybrid network for one patch of 64 × 64 pixels with one channel. Three convolution blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2, turn the patch into 8 × 8 positions of 64 numbers. These 64 positions are the tokens of a small transformer: Put a learned class token of 64 numbers before them, add learned positions for the 65 places, started from small random numbers with a standard deviation of 0.02, and pass the tokens through two transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with one output, the logit of a stenosis. Train the network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then compute the probabilities of the test patches and their AUC against the column "stenosis".  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of trainable weights and the validation AUC of every epoch, and keep the test AUC, with three decimals, under the name "answer".  
> Test: The network must have 94,881 trainable weights, and no patch of a test angiogram may be used for training.  

In [ ]:
#@title Data of step 3.2: run this cell (it prepares the table "data" and shows it)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.2
teach.check_step("Step 3.2", "answer", globals())

In [ ]:
#@title Ready code of step 3.2 (run it only if the code of the assistant does not pass the check)
data = _arcade_table()   # a few minutes the first time: ARCADE is downloaded once and its patches are cut


def standardise(patches):
    patches = np.asarray(patches, dtype=np.float32)
    m = patches.mean(axis=(1, 2), keepdims=True)
    s = patches.std(axis=(1, 2), keepdims=True)
    return (patches - m) / (s + 0.02)


class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        w = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        return self.out((w @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    def __init__(self, dim=64, heads=4, hidden=128):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class Transformer(nn.Module):
    def __init__(self, tokens, depth, dim=64):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, tokens + 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[Block(dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 1)

    def forward(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        return self.head(self.norm(self.blocks(t))[:, 0]).squeeze(1)



def block(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(block(1, 16), block(16, 32), block(32, 64))
        self.transformer = Transformer(64, 2)

    def forward(self, x):
        return self.transformer(self.stem(x).flatten(2).transpose(1, 2))

torch.manual_seed(2026)
net = Net()
print("trainable weights:", sum(p.numel() for p in net.parameters() if p.requires_grad))
sets = data["set"].to_numpy()
X = torch.as_tensor(standardise(np.stack(data["patch"])))[:, None]
y = torch.as_tensor(data["stenosis"].to_numpy(), dtype=torch.float32)
tr_, va_, te_ = [torch.as_tensor(sets == s) for s in ("training", "validation", "test")]


@torch.no_grad()
def probabilities(X_):
    net.eval()
    return torch.cat([torch.sigmoid(net(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xtr, ytr = X[tr_], y[tr_]
torch.manual_seed(2026)
optimiser = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
best, state = -1.0, None
for epoch in range(8):
    net.train()
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        b = order[i:i + 64]
        xb = torch.rot90(Xtr[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xb = torch.flip(xb, dims=(3,))
        loss = loss_fn(net(xb), ytr[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    auc = roc_auc_score(y[va_].numpy(), probabilities(X[va_]))
    print(f"epoch {epoch + 1}: validation AUC {auc:.3f}")
    if auc > best:
        best, state = auc, copy.deepcopy(net.state_dict())
net.load_state_dict(state)
answer = round(float(roc_auc_score(y[te_].numpy(), probabilities(X[te_]))), 3)
print("test AUC:", answer)
teach.check_step("Step 3.2", "answer", globals())

**What the result shows.** The hybrid reaches a test AUC of 0.934 with fewer weights than the vision transformer, and both leave the convolutional network behind. Whether the small difference between the hybrid and the vision transformer is real is a question for a bootstrap of the test angiograms, as in Section 7 of the Python path.

### Step 3.3: Where does the vision transformer look?

**Goal.** Measure how much of the attention of the class token of the vision transformer falls on the centre of the patch, where a stenosis lies.

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know whether the attention of a vision transformer that finds stenoses on angiograms falls where the stenosis is, before reading its attention maps as an explanation.  
> Data: The notebook holds a table named "data" with the 3,088 test patches of 64 × 64 pixels of ARCADE, a public data set of X-ray coronary angiograms. Each row is a patch: "angiogram" gives the number of its angiogram, "stenosis" is 1 for a patch centred on a stenosis outlined by the experts and 0 for a patch on a vessel far from every stenosis, and each cell of "attention" holds a table of 8 × 8 numbers that add up to 1: the attention that the class token of the small vision transformer of the course pays to each of the 64 squares of 8 × 8 pixels of the patch, followed through its four layers (attention rollout). The four central squares cover the central 16 × 16 pixels of the patch, rows and columns 4 and 5 when they are counted from 1.  
> Task: For every patch, add up the attention of the four central squares. Compute the mean of this share over the patches with a stenosis and over the others, and compare both with 4/64 = 0.0625, the share of the four squares if the attention were spread evenly.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two means, and keep the mean of the patches with a stenosis, with three decimals, under the name "answer".  
> Test: Every share must lie between 0 and 1.  

In [ ]:
#@title Data of step 3.3: run this cell (it prepares the table "data" and shows it)
_m, _std, _pred = _course_model("vit")   # about four minutes the first time: the small vision transformer of the course is trained once
_a = _arcade_table()
_a = _a[_a["set"] == "test"].reset_index(drop=True)
_X = _std(np.stack(_a["patch"]))
_m.eval()
_maps = []
with torch.no_grad():
    for _i in range(0, len(_X), 256):
        _m(torch.as_tensor(_X[_i:_i + 256])[:, None])
        for _w in np.stack([_b.attn.weights.mean(1).numpy() for _b in _m.blocks], axis=1):
            _r = np.eye(65)
            for _l in _w:
                _l = 0.5 * _l + 0.5 * np.eye(65)
                _r = (_l / _l.sum(axis=1, keepdims=True)) @ _r
            _maps.append((_r[0, 1:] / _r[0, 1:].sum()).reshape(8, 8))
data = pd.DataFrame({"angiogram": _a["angiogram"], "stenosis": _a["stenosis"], "attention": _maps})
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of step 3.3
teach.check_step("Step 3.3", "answer", globals())

In [ ]:
#@title Ready code of step 3.3 (run it only if the code of the assistant does not pass the check)
_m, _std, _pred = _course_model("vit")   # about four minutes the first time: the small vision transformer of the course is trained once
_a = _arcade_table()
_a = _a[_a["set"] == "test"].reset_index(drop=True)
_X = _std(np.stack(_a["patch"]))
_m.eval()
_maps = []
with torch.no_grad():
    for _i in range(0, len(_X), 256):
        _m(torch.as_tensor(_X[_i:_i + 256])[:, None])
        for _w in np.stack([_b.attn.weights.mean(1).numpy() for _b in _m.blocks], axis=1):
            _r = np.eye(65)
            for _l in _w:
                _l = 0.5 * _l + 0.5 * np.eye(65)
                _r = (_l / _l.sum(axis=1, keepdims=True)) @ _r
            _maps.append((_r[0, 1:] / _r[0, 1:].sum()).reshape(8, 8))
data = pd.DataFrame({"angiogram": _a["angiogram"], "stenosis": _a["stenosis"], "attention": _maps})
shares = np.array([m[3:5, 3:5].sum() for m in data["attention"]])
y = data["stenosis"].to_numpy()
print("with a stenosis", round(shares[y == 1].mean(), 3), "| others", round(shares[y == 0].mean(), 3), "| even", 4 / 64)
answer = round(float(shares[y == 1].mean()), 3)
teach.check_step("Step 3.3", "answer", globals())

**What the result shows.** On the patches with a stenosis, the class token sends a share of 0.153 of its attention to the four central squares, which cover a sixteenth of the patch. The map points to the centre more than an even spread would, but it shows where information flows, not why the network decides.

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#ai-pair-programming-part-7-reviewing-an-image-pipeline) and continue reading at «Working with an AI assistant, part 7: Reviewing an image pipeline». The weekly project can also be done on the prompt path: Its prompts are in the section «Weekly project on the prompt path» below.

## Weekly project on the prompt path

*Project of the week: [README of Week 7, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-07/README.md#weekly-project)*

The weekly project can also be built with the prompts below, one for each step. Run the preparation cell once: It loads the data of the project. Each step then works like a step of the stops, with one difference: The project has no check cell and no ready code, because its results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

In [ ]:
#@title Preparation of the project: run this cell once (it loads the data of the project; if Section 0 has not run in this session, it runs it first)
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
_SETS = {"training": "training", "validation": "validation", "test": "test", "train": "training", "val": "validation"}


def _synthetic_table():
    """Patches of the synthetic angiograms of 300 training, 100 validation and 100 test patients of the virtual clinic."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"set": _SETS[s], "patient": ids, "patch": list(X), "stenosis": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """Patches of the 1500 angiograms of ARCADE (stenosis set) with their set and angiogram."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"set": _SETS[s], "angiogram": ids, "patch": list(X), "stenosis": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """A network of the course, trained once on the training patches and kept for the next steps (one to four minutes)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Every patch on its own: mean grey value 0 and standard deviation 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """Convolution of 3 x 3, batch normalisation, ReLU and max pooling of 2 x 2: half the size, b channels."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Four convolution blocks (16, 32, 64 and 64 channels), the mean of every channel and one output."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Self-attention with several heads; the weights of the last call are kept for the maps."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """The number of trainable weights of a network."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """A hybrid: three convolution blocks turn the patch into 8 x 8 positions of 64 numbers, the tokens of two transformer blocks."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """The probability of a stenosis for every patch."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001, groups of 64 patches turned by a random quarter turn and mirrored half of the time; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["set"].to_numpy()
    X, y = standardise(np.stack(t["patch"])), t["stenosis"].to_numpy()
    tr_, va_ = sets == "training", sets == "validation"
    torch.manual_seed(2026)
    model = {"synthetic_cnn": SmallCNN, "cnn": SmallCNN, "vit": SmallViT, "hybrid": Hybrid}[kind]()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]



def _probability_table():
    """Validation and test patches of ARCADE with the probabilities of the three networks of the course (about seven minutes the first time)."""
    t = _arcade_table()
    t = t[t["set"] != "training"].reset_index(drop=True)
    out = t[["set", "angiogram", "stenosis"]].copy()
    for kind, name in (("cnn", "convolutional network"), ("vit", "vision transformer"), ("hybrid", "hybrid")):
        m, std, pred = _course_model(kind)
        out[name] = pred(m, std(np.stack(t["patch"])))
    return out


def _angiogram_table():
    """The 1,500 angiograms of ARCADE with their outlined stenoses, as grey values from 0 to 255."""
    if "_angio_cache" not in globals():
        arcade.download()
        rows = []
        for s in arcade.SPLITS:
            for a in arcade.annotations(s):
                rows.append({"set": _SETS[s], "angiogram": a["id"], "image": (arcade.load(s, a["file"]) * 255).round().astype(np.uint8),
                             "stenoses": [(round(x["x"]), round(x["y"])) for x in a["stenoses"]]})
        globals()["_angio_cache"] = pd.DataFrame(rows)
    return _angio_cache


def _detector_table():
    """The test angiograms with the probabilities of the detector of the course every 16 pixels and its threshold (about fifteen minutes the first time)."""
    if "_detector_cache" in globals():
        return _detector_cache
    def standardise(X):
        """Every patch on its own: mean grey value 0 and standard deviation 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """Convolution of 3 x 3, batch normalisation, ReLU and max pooling of 2 x 2: half the size, b channels."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Four convolution blocks (16, 32, 64 and 64 channels), the mean of every channel and one output."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Self-attention with several heads; the weights of the last call are kept for the maps."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """The number of trainable weights of a network."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """A hybrid: three convolution blocks turn the patch into 8 x 8 positions of 64 numbers, the tokens of two transformer blocks."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """The probability of a stenosis for every patch."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam with a learning rate of 0.001, groups of 64 patches turned by a random quarter turn and mirrored half of the time; the weights of the epoch with the best validation AUC are kept."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    def heat_map(model, image, size=64, stride=32):
        """The probability of a stenosis for windows of 64 x 64 pixels every 32 pixels: a grid of 16 x 16 for an angiogram of 512 x 512."""
        h = size // 2
        pad = np.pad(image, h, mode="reflect")
        centres = np.arange(stride // 2, image.shape[0], stride)
        windows = [pad[r:r + size, c:c + size] for r in centres for c in centres]
        return predict(model, standardise(windows)).reshape(len(centres), len(centres)), centres


    def lesions_found(grid, centres, stenoses, threshold, near=32):
        """A stenosis is found when a window whose centre lies within 32 pixels of it reaches the threshold; an alarm is a window over the threshold farther than 32 pixels from every stenosis."""
        cy, cx = np.meshgrid(centres, centres, indexing="ij")
        hot = grid >= threshold
        found, alarm = 0, hot.copy()
        for s in stenoses:
            close = np.hypot(cx - s["x"], cy - s["y"]) <= near
            found += int((hot & close).any())
            alarm &= ~close
        return found, int(alarm.sum())
    from scipy.ndimage import maximum_filter
    data_ = {s: arcade.patches(s, per_stenosis=4, neg_per_pos=1.5, background=1.5, jitter=16) for s in arcade.SPLITS}
    torch.manual_seed(2026)
    net = Hybrid()
    train(net, standardise(data_["train"][0]), data_["train"][1], standardise(data_["val"][0]), data_["val"][1], epochs=8, verbose=False)

    def scan(split):
        return [(a, heat_map(net, arcade.load(split, a["file"]), stride=16)) for a in arcade.annotations(split)]

    def alarms(grid, centres, stenoses, t):
        cy, cx = np.meshgrid(centres, centres, indexing="ij")
        peaks = (grid >= t) & (grid == maximum_filter(grid, size=5))
        far = np.ones(peaks.sum(), dtype=bool)
        for s in stenoses:
            far &= np.hypot(cx[peaks] - s["x"], cy[peaks] - s["y"]) > 32
        return int(far.sum())

    val = scan("val")
    thr = min(t for t in np.arange(0.5, 1.0, 0.01) if np.mean([alarms(g, c, a["stenoses"], t) for a, (g, c) in val]) <= 1.0)
    test = scan("test")
    globals()["_detector_cache"] = pd.DataFrame({"angiogram": [a["id"] for a, _ in test], "image": [(arcade.load("test", a["file"]) * 255).round().astype(np.uint8) for a, _ in test],
                                                 "stenoses": [[(round(x["x"]), round(x["y"])) for x in a["stenoses"]] for a, _ in test],
                                                 "probabilities": [g for _, (g, c) in test], "threshold": round(float(thr), 2)})
    return _detector_cache
print("The project is ready.")

### Project step 1: Three networks, one test

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to compare a convolutional network, a vision transformer and a hybrid for finding stenoses on real coronary angiograms, trained in the same way and tested on the same angiograms.  
> Data: The notebook holds a table named "data" with patches of 64 × 64 pixels cut from the 1,500 X-ray coronary angiograms of the stenosis set of ARCADE, a public data set (training 1,000 angiograms, validation 200, test 300). Each row is a patch: "set" gives its set ("training", "validation" or "test"), "angiogram" the number of its angiogram, and each cell of "patch" holds the patch as an image of 64 × 64 pixels with grey values from 0 to 1. "stenosis" is 1 when the patch is centred within 6 pixels of the centre of a stenosis outlined by the experts and 0 when it lies on a vessel at least 48 pixels from every outlined stenosis and from the border of the image.  
> Task: Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build three networks. First: Build a small convolutional network for one patch of 64 × 64 pixels with one channel: four blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32, 64 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2; then the mean of every channel over its positions and a linear layer with one output, the logit of a stenosis. Second: Build a small vision transformer for one patch of 64 × 64 pixels with one channel. Cut the patch into 64 squares of 8 × 8 pixels and map every square to 64 numbers with the same learned linear map. Put a learned class token of 64 numbers before the 64 tokens and add learned positions, one vector of 64 numbers for each of the 65 places, started from small random numbers with a standard deviation of 0.02. Pass the tokens through four transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with one output, the logit of a stenosis. Third: Build a hybrid network for one patch of 64 × 64 pixels with one channel. Three convolution blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2, turn the patch into 8 × 8 positions of 64 numbers. These 64 positions are the tokens of a small transformer: Put a learned class token of 64 numbers before them, add learned positions for the 65 places, started from small random numbers with a standard deviation of 0.02, and pass the tokens through two transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with one output, the logit of a stenosis. Train each network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then compute the probabilities of the test patches and their AUC against the column "stenosis". Draw 2,000 bootstrap samples of the 300 test angiograms with replacement, keeping all patches of every angiogram drawn, and give every AUC a 95 percent interval from the 2.5th to the 97.5th percentile. Fix the random seed to 2026 so that the result can be repeated. In the same samples, compute the difference of the AUC of the vision transformer and of the hybrid to that of the convolutional network.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each network, its trainable weights, its training time and its test AUC with the interval, and the two differences with their intervals; keep the largest test AUC, with three decimals, under the name "answer".  
> Test: The three networks must have 60,641, 142,465 and 94,881 trainable weights, and the intervals must come from angiograms, not from patches.  

In [ ]:
#@title Data of project step 1: run this cell (it prepares the table "data" and shows it)
data = _arcade_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: A threshold for the laboratory

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know what the networks for stenoses on angiograms would give in practice at a fixed threshold, among curated patches and among the windows of a whole angiogram, where a stenosis is rare.  
> Data: The notebook holds a table named "data" with the 6,336 validation and test patches of 64 × 64 pixels of ARCADE, a public data set of X-ray coronary angiograms (validation 200 angiograms, test 300). Each row is a patch: "set" gives its set ("validation" or "test"), "angiogram" the number of its angiogram, "stenosis" is 1 for a patch centred on a stenosis outlined by the experts and 0 for a patch on a vessel far from every stenosis, and the three columns "convolutional network", "vision transformer" and "hybrid" give the probability of a stenosis from the three small networks of the course, trained on the training patches.  
> Task: For each of the three networks, choose on the patches of the set "validation" the threshold that 90 percent of the patches with a stenosis reach. Apply it to the patches of the set "test" and compute the sensitivity, the specificity and the positive predictive value. Then compute the positive predictive value that the same sensitivity and specificity would give at a prevalence of 2 percent: the sensitivity times 0.02, divided by itself plus one minus the specificity times 0.98.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each network, its threshold and the four values, and keep the positive predictive value of the hybrid at a prevalence of 2 percent, with three decimals, under the name "answer".  
> Test: The thresholds must be chosen on the validation patches only, and every value must lie between 0 and 1.  

In [ ]:
#@title Data of project step 2: run this cell (it prepares the table "data" and shows it)
data = _probability_table()   # about seven minutes the first time: the three networks of the course are trained once
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: A detector for whole angiograms

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants a program that marks the stenoses on whole coronary angiograms, with about one false detection per angiogram, and wants to know which share of the stenoses it finds.  
> Data: The notebook holds a table named "data" with the 1,500 X-ray coronary angiograms of the stenosis set of ARCADE, a public data set (training 1,000, validation 200, test 300). Each row is an angiogram: "set" gives its set ("training", "validation" or "test"), "angiogram" its number, each cell of "image" holds the angiogram as an image of 512 × 512 pixels with grey values from 0 to 255, and "stenoses" lists the centres of the stenoses outlined by the experts, each as (column, row) in pixels counted from 0.  
> Task: Bring every image to grey values from 0 to 1. From every angiogram cut windows of 64 × 64 pixels: four for every stenosis, centred on it and shifted at random by up to 16 pixels in each direction (label 1); and, for every stenosis, six windows centred on dark thin structures at least 48 pixels from the border of the image, among the pixels with the largest 3 percent of the values of a black top-hat with a disk of radius 6, and six windows at random places, all at least 48 pixels from every stenosis (label 0). Standardise every patch on its own: Subtract the mean of its grey values and divide by their standard deviation plus 0.02. Build a hybrid network for one patch of 64 × 64 pixels with one channel. Three convolution blocks, each a convolution with filters of 3 × 3 pixels and a padding of one pixel (16, 32 and 64 filters), batch normalisation, ReLU and max pooling of 2 × 2, turn the patch into 8 × 8 positions of 64 numbers. These 64 positions are the tokens of a small transformer: Put a learned class token of 64 numbers before them, add learned positions for the 65 places, started from small random numbers with a standard deviation of 0.02, and pass the tokens through two transformer blocks. In each block, layer normalisation is followed by self-attention with four heads, whose output is added to the input of the block; a second layer normalisation is followed by a feed-forward network with a hidden layer of 128 units and the GELU activation, whose output is added again. At the end, take the class token, apply layer normalisation and a linear layer with one output, the logit of a stenosis. Train the network on the training patches for 8 epochs, in shuffled groups of 64 patches, with the binary cross-entropy loss on the logit and the Adam optimiser with a learning rate of 0.001. As data augmentation, turn every group by a random quarter turn and mirror it from left to right with a probability of one half. After every epoch, compute the AUC of the probabilities of the validation patches, and at the end keep the weights of the epoch with the largest validation AUC. Fix the random seed to 2026 so that the result can be repeated. Then scan every validation and test angiogram with windows of 64 × 64 pixels whose centres lie every 16 pixels, starting at 8, with the image extended at its borders by reflection, and compute the probability of every window. A detection is a window whose probability is at least the threshold and is the largest among the windows of its neighbourhood of 5 × 5 centres. A stenosis is found when a detection lies within 32 pixels of it, and every other detection is false. Choose the smallest threshold from 0.50 to 0.99 in steps of 0.01 that gives at most one false detection per angiogram on the validation angiograms, and apply it to the test angiograms.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the chosen threshold. Then show a table with one row for each of the thresholds 0.50, 0.70, 0.90 and the chosen one: the number of stenoses found out of all stenoses of the test angiograms and the false detections per test angiogram. Keep the share of the stenoses found at the chosen threshold, with three decimals, under the name "answer".  
> Test: The threshold must be chosen on the validation angiograms only, and the test angiograms must hold 386 stenoses.  

In [ ]:
#@title Data of project step 3: run this cell (it prepares the table "data" and shows it)
data = _angiogram_table()
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: The errors read by a cardiologist

**Prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see the stenoses that a detector misses and the places that it marks wrongly, to judge what such a tool could do in the catheterisation laboratory.  
> Data: The notebook holds a table named "data" with the 300 test angiograms of ARCADE, a public data set of X-ray coronary angiograms. Each row is an angiogram: "angiogram" gives its number, each cell of "image" holds it as an image of 512 × 512 pixels with grey values from 0 to 255, "stenoses" lists the centres of the stenoses outlined by the experts as (column, row) in pixels counted from 0, each cell of "probabilities" holds a table of 32 × 32 probabilities of a stenosis from the detector of the course, one for every window of 64 × 64 pixels whose centre lies at 8, 24, 40 and so on up to 504 pixels in each direction, and "threshold" holds the threshold of the detector, chosen on the validation angiograms for at most one false detection per angiogram.  
> Task: A detection is a window whose probability is at least the threshold and is the largest among the windows of its neighbourhood of 5 × 5 centres. A stenosis is found when a detection lies within 32 pixels of it, and every other detection is false. For every angiogram, list the missed stenoses and the false detections with their probability. Then show five missed stenoses and the five false detections with the largest probabilities, each as the window of 128 × 128 pixels around it with the probabilities drawn over the image, the stenoses of the experts as circles and the detections as squares.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the ten windows, print the number of missed stenoses and of false detections, and keep the number of missed stenoses under the name "answer".  
> Test: The missed stenoses and the found ones must add up to 386.  

In [ ]:
#@title Data of project step 4: run this cell (it prepares the table "data" and shows it)
data = _detector_table()   # about fifteen minutes the first time: the detector of the course is trained once and scans the angiograms
globals().pop("answer", None)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

<a id="part-2"></a>

# Part 2: Python path

**For whom?** This part is for participants who want to learn to write Python. Part 1 is not needed for it: After Section 0, start here.

**How it works.** Run the cells from top to bottom with Shift+Enter. The *Python step* at the start teaches the parts of the language that the week needs. In the numbered sections, exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. Cells marked *Advanced (optional)* are extensions for those who already program. A participant who opens this part in the middle runs the *Catch-up* cell at the start of that section. Some exercise cells begin with a line that prepares a table named `data`: It serves the prompt of the exercise in the prompt workshop of the interactive lab, and on the Python path the exercise can be solved with the tables named in its text.

## Python step 7: Images as arrays and the arithmetic of attention

This week the data are images, and the networks compare every part of an image with every other part. This step shows how an image sits in an array of numbers, how a window is cut from it, how an image is cut into patches with `reshape` and `transpose`, how the matrix product `@` and the softmax compute attention for a whole table of tokens at once, and how an array becomes a tensor that a network can read.

In [ ]:
#@title Preparation of the Python step: run this cell once (if Section 0 has not run in this session, it runs it)
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("Section 0 has not run in this session. It runs now and prepares the course files, the packages and the expected answers; this can take a minute.")
    _catch_up(1, "Section 0 is ready.")
print("The Python step is ready.")

In [ ]:
# Python practice: a small example for learning the language
views = angiogram.patient_frames("VC-2427")
left, truth = views["left"]
print(left.shape, left.dtype, left.min(), left.max())
image = left / 255
lad = truth["stenoses"][0]
x, y = int(lad["x"]), int(lad["y"])
window = image[y - 32:y + 32, x - 32:x + 32]
print(lad["vessel"], lad["stenosis"], "%", "| window", window.shape, "| mean grey value", round(float(window.mean()), 3))
fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
ax[0].imshow(image, cmap="gray", vmin=0, vmax=1)
ax[0].add_patch(plt.Rectangle((x - 32, y - 32), 64, 64, fill=False, color="#9B1B1B"))
ax[1].imshow(window, cmap="gray", vmin=0, vmax=1)
for a in ax:
    a.axis("off")
plt.show()

**Python note.** `angiogram.patient_frames` draws the two angiograms of a patient of the virtual clinic and returns a dictionary with the keys `"left"` and `"right"`; each value is a pair of the image and its truth, and `left, truth = ...` gives both a name. The image is an array of 512 × 512 whole numbers from 0 to 255 (`uint8`), and dividing by 255 gives grey values from 0 to 1. Rows come first: `image[y - 32:y + 32, x - 32:x + 32]` takes 64 rows around the row `y` and 64 columns around the column `x`, a window of 64 × 64 pixels around the stenosis. `plt.Rectangle` draws a box whose corner and size are given in pixels.

In [ ]:
def to_patches(image, p):
    """Cut a grey image into squares of p x p pixels, row by row; each square becomes a row of p * p numbers."""
    h, w = image.shape[0] // p * p, image.shape[1] // p * p
    return image[:h, :w].reshape(h // p, p, w // p, p).transpose(0, 2, 1, 3).reshape(-1, p * p)

# Python practice: cutting into patches
a = np.arange(16).reshape(4, 4)
print(a)
print(to_patches(a, 2))
small = np.asarray(Image.fromarray(left).resize((224, 224), Image.BILINEAR)) / 255
rows = to_patches(small, 16)
print(small.shape, "→", rows.shape)

**Python note.** `np.arange(16).reshape(4, 4)` writes the numbers 0 to 15 into 4 rows of 4. `to_patches` cuts an image into squares in four moves: `reshape(h // p, p, w // p, p)` gives every pixel four indices (the row of its patch, its row inside the patch, the column of its patch, its column inside the patch); `transpose(0, 2, 1, 3)` brings the two indices of the patch to the front; the last `reshape` turns every patch into a row of $p \times p$ numbers. The first patch of `a` is 0, 1, 4, 5: the upper left square. `Image.fromarray(...).resize((224, 224), Image.BILINEAR)` shrinks the angiogram to 224 × 224 pixels, the size that ViT-B/16 expects, and 224/16 = 14 patches in each direction give 196 rows of 256 numbers.

In [ ]:
def softmax(scores, axis=-1):
    """Turn scores into weights that are positive and add up to 1 along an axis."""
    e = np.exp(scores - scores.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """Scaled dot-product attention: every query takes a weighted mean of the values, with weights from its similarity to the keys."""
    weights = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return weights @ V, weights

# Python practice: the matrix product and the softmax
Q = np.array([[1.0, 0.0]])
K = np.array([[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
scores = Q @ K.T / np.sqrt(2)
print("scores", np.round(scores, 3))
output, weights = attention(Q, K, V)
print("weights", np.round(weights, 3), "| sum", weights.sum())
print("output", np.round(output, 3))

**Python note.** `@` is the matrix product: `Q @ K.T` multiplies every query, a row of `Q`, with every key, a row of `K` turned into a column by `.T`, and gives a table with one row per query and one column per key. `softmax` subtracts the largest score of each row before `np.exp`, which changes nothing in the result but keeps large scores from overflowing; `keepdims=True` keeps the sums as a column, so that every row is divided by its own sum. `attention` returns two things, the output and the weights, and `output, weights = ...` names both. The numbers are those of the worked example of the lecture page.

In [ ]:
# Python practice: from an array to a tensor
x = torch.as_tensor(small, dtype=torch.float32)[None, None]
print(x.shape, "→", x.repeat(1, 3, 1, 1).shape)
embed = nn.Linear(256, 768)
print("weights of the linear map:", sum(p.numel() for p in embed.parameters()))
with torch.no_grad():
    tokens = embed(torch.as_tensor(rows, dtype=torch.float32))
print("tokens:", tuple(tokens.shape))

**Python note.** A network reads tensors, the arrays of PyTorch, with the images of a batch along the first axis and the channels along the second: `[None, None]` adds both, so one grey image of 224 × 224 becomes a tensor of 1 × 1 × 224 × 224, and `.repeat(1, 3, 1, 1)` copies the grey channel three times for a network that expects colour. `nn.Linear(256, 768)` is a linear map from 256 numbers to 768: a table of 256 × 768 weights and 768 biases, 197,376 numbers. Applied to the 196 rows of the patches, it gives 196 tokens of 768 numbers, which is the patch embedding of a vision transformer for one grey channel. `torch.no_grad()` computes without recording what training would need.

**Quick check.** An image has the shape (512, 512). What is the shape of `image[100:164, 200:264]`?

<details><summary>Answer</summary>

(64, 64): rows 100 to 163 and columns 200 to 263. The end of a slice is not included.

</details>

**Exercise 7.1.** Into how many patches of 16 × 16 pixels does `to_patches` cut the left angiogram of the case after it is shrunk to 224 × 224 pixels (`small`)? Keep the number of patches.

In [ ]:
data = exercise_table("7.1")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 7.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = to_patches(small, 16).shape[0]
teach.check("Exercise 7.1", answer)

**Exercise 7.2.** With the keys and values of the worked example, what is the attention weight of the third token for the query (0, 1)? Keep it with three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 7.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
_, w = attention(np.array([[0.0, 1.0]]), K, V)
answer = round(float(w[0, 2]), 3)
teach.check("Exercise 7.2", answer)

**Exercise 7.3.** What is the mean grey value, from 0 to 1, of the window of 64 × 64 pixels centred on the stenosis of the circumflex artery in the left angiogram of the case? The stenosis is `truth["stenoses"][1]`. Keep it with three decimals.

In [ ]:
data = exercise_table("7.3")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise 7.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s = truth["stenoses"][1]
x1, y1 = int(s["x"]), int(s["y"])
answer = round(float(image[y1 - 32:y1 + 32, x1 - 32:x1 + 32].mean()), 3)
teach.check("Exercise 7.3", answer)

## 1. The angiograms of the virtual clinic

*Lecture: [Images of the catheterisation laboratory: The angiograms of the virtual clinic](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#images-of-the-catheterisation-laboratory-the-angiograms-of-the-virtual-clinic)*

Every patient of the registry with an angiography has two synthetic angiograms, a view of the left coronary artery and a view of the right one, drawn from the lesions that the reports of Week 6 describe. The truth of every frame lists its stenoses with their vessel, segment, degree, centre and box.

**From the formulas to the code.** A significant stenosis is 50 percent or more in the left main coronary artery and 70 percent or more elsewhere: `coronary.significant(territory, stenosis)`. The view of a vessel: `angiogram.VIEW_OF[vessel]`, `"left"` or `"right"`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(1, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
patients, lesions = angiogram.registry()
split = patients["split"].to_numpy()
CASE = "VC-2427"
case_views = angiogram.patient_frames(CASE)

print(patients["split"].value_counts())
print(lesions[lesions["patient_id"] == CASE][["vessel", "segment", "stenosis", "kind"]])
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, (view, (img, t)) in zip(axes, case_views.items()):
    ax.imshow(img, cmap="gray")
    for s in t["stenoses"]:
        bx, by, bw, bh = s["box"]
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, lw=1.5, color="#9B1B1B" if s["significant"] else "#B97813"))
        ax.text(bx, by - 6, f"{s['vessel']} {s['stenosis']}%", color="#9B1B1B" if s["significant"] else "#B97813", fontsize=9)
    ax.set_title(view)
    ax.axis("off")
plt.show()

**Python note.** `angiogram.registry()` returns two tables: `patients`, one row per patient with an angiography, with its set, and `lesions`, one row per lesion, with the vessel, the segment, the degree in percent and the kind. The sets are those of Week 6, by the number of the patient. `case_views.items()` gives the pairs (view, frame), and `zip` walks through them together with the two axes of the figure. In the truth of a frame, `box` is (x, y, width, height) of a square around the stenosis, in pixels.

In [ ]:
sig = [int(coronary.significant(t, s)) for t, s in zip(lesions["territory"], lesions["stenosis"])]
table = lesions.assign(significant=sig, view=lesions["vessel"].map(angiogram.VIEW_OF)).merge(patients[["patient_id", "split"]], on="patient_id")
print(pd.crosstab(table["split"], [table["view"], table["significant"]]))

**Observe and reflect.** Look at the two angiograms of the case. Which stenosis would you call significant by eye, and which detail of a real angiogram does the synthetic image leave out?

**Exercise A.** How many of the 394 patients of the test set have at least one significant stenosis in a vessel of the left coronary view? Use `table` of the cell above.

In [ ]:
data = exercise_table("A")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(table[(table["split"] == "test") & (table["view"] == "left") & (table["significant"] == 1)]["patient_id"].nunique())
teach.check("Exercise A", answer)

## 2. Attention by hand

*Lecture: [Self-attention: The relation between different regions of the image](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#self-attention-the-relation-between-different-regions-of-the-image)*

Attention compares a query with every key, turns the scores into weights with a softmax and passes on the weighted mean of the values. In self-attention, the queries, the keys and the values are linear maps of the same tokens.

**From the formulas to the code.** $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})\,V$: `softmax(Q @ K.T / np.sqrt(K.shape[1])) @ V` in `attention`. Self-attention: `Q = X @ Wq`, `K = X @ Wk`, `V = X @ Wv`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(2, "Catch-up finished: the code of the earlier sections has run.")

**How far does a convolutional network see?** *Lecture: [The limits of CNNs: Lack of global context](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#from-convolution-to-attention)*

A position of a convolutional layer sees only a part of the image, its *receptive field*. Each convolution of 3 × 3 pixels widens the field by two steps of the layer below, and each pooling of 2 × 2 doubles the step. The cell follows the small convolutional network of this week, used from Section 5 on, through its eight layers.

**From the formulas to the code.** $r_l = r_{l-1} + (k_l - 1)\,j_{l-1}$ and $j_l = j_{l-1}\,s_l$: `field += (kernel - 1) * jump` and `jump *= stride` in `receptive_field`.

In [ ]:
def receptive_field(layers):
    """The receptive field after every layer of a stack of (name, kernel, stride): it grows by (kernel - 1) times the product of the strides of the layers before."""
    field, jump, rows = 1, 1, []
    for name, kernel, stride in layers:
        field += (kernel - 1) * jump
        jump *= stride
        rows.append({"layer": name, "receptive field (pixels)": field, "step between positions": jump})
    return pd.DataFrame(rows)


SMALL_CNN_LAYERS = [("convolution 1", 3, 1), ("pooling 1", 2, 2), ("convolution 2", 3, 1), ("pooling 2", 2, 2),
                    ("convolution 3", 3, 1), ("pooling 3", 2, 2), ("convolution 4", 3, 1), ("pooling 4", 2, 2)]

print(receptive_field(SMALL_CNN_LAYERS).to_string(index=False))
layers = int(np.ceil((512 - 1) / 2))
print("convolutions of 3 x 3 without pooling for a field of 512 pixels:", layers)

**Python note.** `receptive_field` walks through the layers in order and keeps two numbers: the width of the field in pixels of the input and the step between two neighbouring positions of the layer. `int(np.ceil(...))` rounds up to a whole number of layers. The last positions of the small network see 46 × 46 pixels of a patch of 64 × 64; only the mean over all positions at the end brings the whole patch together.

In [ ]:
def softmax(scores, axis=-1):
    """Turn scores into weights that are positive and add up to 1 along an axis."""
    e = np.exp(scores - scores.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """Scaled dot-product attention: every query takes a weighted mean of the values, with weights from its similarity to the keys."""
    weights = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return weights @ V, weights

for angle in (0, 45, 90, 180):
    q = np.array([[np.cos(np.radians(angle)), np.sin(np.radians(angle))]])
    _, w = attention(q, K, V)
    print(f"query at {angle:3}°: weights {np.round(w[0], 3)}")

**Python note.** `np.radians` turns degrees into radians, and `np.cos` and `np.sin` give a query of length 1 that points in that direction. At 0° the query points like the key of the first token and gives it the largest weight; at 90° it points like the key of the second; at 180° it points away from both, and the weights move to the third token, whose key is the least opposed.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X = to_patches(small, 16)[[40, 41, 54, 120, 190]]          # five patches of the angiogram
Wq, Wk, Wv = (rng.normal(0, 0.05, size=(256, 16)) for _ in range(3))
out, w = attention(X @ Wq, X @ Wk, X @ Wv)
print(np.round(w, 2))
plt.figure(figsize=(3.2, 2.8))
plt.imshow(w, cmap="Oranges", vmin=0, vmax=1)
plt.colorbar()
plt.xlabel("key")
plt.ylabel("query")
plt.show()

**Python note.** Five patches of the angiogram, each a row of 256 grey values, become five tokens. Three matrices of 256 × 16 random numbers map them to queries, keys and values of 16 numbers; a network would learn these matrices. The table of weights has one row per query, and every row adds up to 1. With random matrices the weights reflect only how alike the patches are: Patches with a similar brightness attend to each other.

**Exercise B.** Without the division by the square root of the length of the keys, what weight does the first token of the worked example get for the query (1, 0)? Keep it with three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(softmax(np.array([[1.0, 0.0]]) @ K.T)[0, 0]), 3)
teach.check("Exercise B", answer)

## 3. Patches and the vision transformer ViT-B/16

*Lecture: [Patch embedding: Splitting the image into 16×16 patches](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#the-vision-transformer-an-image-as-a-sequence-of-patches)*

ViT-B/16 cuts an image of 224 × 224 pixels into 196 patches of 16 × 16 pixels, maps each to a token of 768 numbers, puts a class token before them and adds learned positions; 12 transformer blocks follow. Without its last layer it returns the class token of the last block.

**From the formulas to the code.** $N = (224/16)^2 = 196$: the 196 rows of `to_patches(small, 16)`. The patch embedding $E$ is `vit.conv_proj`, the class token $x_{\mathrm{cls}}$ is `vit.class_token`, the positions $E_{\mathrm{pos}}$ are `vit.encoder.pos_embedding` and the 12 blocks are `vit.encoder.layers`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(3, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).eval()
vit.heads = nn.Identity()      # without its last layer, the network returns the class token: 768 numbers per image
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def to_imagenet(images, size=224):
    """Grey images with values from 0 to 1 → tensors of 3 x 224 x 224 with the mean and the standard deviation of ImageNet."""
    x = torch.as_tensor(np.asarray(images, dtype=np.float32))[:, None]
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD


@torch.no_grad()
def vit_features(images, batch=16):
    """The class token of ViT-B/16 for every image: an array of n x 768 numbers."""
    return torch.cat([vit(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


@torch.no_grad()
def vit_attention(image):
    """The attention of the 12 layers of ViT-B/16 for one image, as the mean of the 12 heads: an array of 12 x 197 x 197."""
    x = vit._process_input(to_imagenet([image]))
    x = torch.cat([vit.class_token, x], dim=1) + vit.encoder.pos_embedding
    maps = []
    for block in vit.encoder.layers:
        y = block.ln_1(x)
        out, w = block.self_attention(y, y, y, need_weights=True, average_attn_weights=True)
        maps.append(w[0].numpy())
        x = x + out
        x = x + block.mlp(block.ln_2(x))
    return np.stack(maps)


def rollout(maps):
    """Attention rollout: the attention of each layer, with the residual connection added, multiplied from the first layer to the last."""
    r = np.eye(maps.shape[-1])
    for a in maps:
        a = 0.5 * a + 0.5 * np.eye(len(a))
        r = (a / a.sum(axis=1, keepdims=True)) @ r
    return r

parts = {"patch embedding": vit.conv_proj, "12 blocks": vit.encoder.layers, "final normalisation": vit.encoder.ln}
for name, part in parts.items():
    print(f"{name:22} {sum(p.numel() for p in part.parameters()):>12,}")
counts = {"class token": vit.class_token.numel(),
          "positions": vit.encoder.pos_embedding.numel(),
          "total": sum(p.numel() for p in vit.parameters())}
for name, n_params in counts.items():
    print(f"{name:22} {n_params:>12,}")
with torch.no_grad():
    print("class token of the case:", vit(to_imagenet([image])).shape)

**Python note.** `vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)` builds the network and loads the weights learned on ImageNet, downloaded in Section 0; `.eval()` switches off what only training needs. Replacing `vit.heads`, the last layer of 1,000 classes, by `nn.Identity()`, a layer that passes its input on unchanged, makes the network return the class token. `to_imagenet` brings grey images to the input that the network learned from: 224 × 224 pixels, three equal colour channels and the mean and standard deviation of ImageNet. The total, 85,798,656, lacks the 769,000 weights of the removed last layer.

In [ ]:
maps = vit_attention(image)
roll = rollout(maps)
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
ax[0].imshow(small, cmap="gray")
for a, (title, m) in zip(ax[1:], (("layer 1", maps[0]), ("layer 12", maps[11]), ("rollout", roll))):
    a.imshow(small, cmap="gray")
    a.imshow(np.kron(m[0, 1:].reshape(14, 14), np.ones((16, 16))), cmap="inferno", alpha=0.55)
    a.set_title(title)
for a in ax:
    a.axis("off")
plt.show()

**Python note.** `vit_attention` runs the 12 blocks of the network one by one and asks each for its attention weights, the mean of its 12 heads: an array of 12 × 197 × 197. Row 0 is the class token, and `m[0, 1:]` its attention over the 196 patches, which `reshape(14, 14)` puts back into the grid of the image. `np.kron` with a block of ones enlarges every value to 16 × 16 pixels for the drawing. `rollout` multiplies the layers as in the lecture page.

**Observe and reflect.** The network never saw an angiogram. On which structures does its attention gather in the last layer and in the rollout, and why would a network trained on photographs attend there?

**Exercise C.** How many weights does the first transformer block of ViT-B/16, `vit.encoder.layers[0]`, hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in vit.encoder.layers[0].parameters())
teach.check("Exercise C", answer)

**A vision transformer built with PyTorch.** *Lecture: [Building a ViT with PyTorch; a ready ViT model from Hugging Face](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#building-a-vit-with-pytorch-a-ready-vit-model-from-hugging-face)*

The classes below build a small vision transformer piece by piece, the same plan as ViT-B/16 at a smaller scale: a patch embedding, a class token, learned positions, transformer blocks and a last layer. The mini project of Section 4 trains it on the photographs of CIFAR-10, and Sections 5 to 8 train it on patches of angiograms.

**From the formulas to the code.** $x_i E$ for every patch: `nn.Conv2d(channels, dim, patch, stride=patch)`, a convolution whose filter is as large as its step, so that it maps every square once. $x_{\mathrm{cls}}$: `self.cls`. $E_{\mathrm{pos}}$: `self.pos`. The block of the lecture, $X' = X + \mathrm{MHA}(\mathrm{LN}(X))$ and $X'' = X' + \mathrm{MLP}(\mathrm{LN}(X'))$: the two lines of `Block.forward`.

In [ ]:
class Attention(nn.Module):
    """Self-attention with several heads; the weights of the last call are kept for the maps."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """A transformer block: attention and a small two-layer network, each after layer normalisation and with a residual connection."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """A small vision transformer: squares of 8 x 8 pixels as tokens, a class token, learned positions and four blocks."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # one linear map for every square
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """The number of trainable weights of a network."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.manual_seed(RANDOM_STATE)
mini_vit = SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3, outputs=10)
parts = {"patch embedding": mini_vit.embed, "class token": [mini_vit.cls], "positions": [mini_vit.pos],
         "4 blocks": mini_vit.blocks, "final normalisation": mini_vit.norm, "last layer": mini_vit.head}
for name, part in parts.items():
    weights = part if isinstance(part, list) else list(part.parameters())
    print(f"{name:24} {sum(w.numel() for w in weights):>8,}")
print(f"{'total':24} {n_parameters(mini_vit):>8,}")
x = torch.randn(2, 3, 32, 32)
tokens = mini_vit.embed(x).flatten(2).transpose(1, 2)
print("2 images", tuple(x.shape), "→ tokens", tuple(tokens.shape), "→ scores of 10 classes", tuple(mini_vit(x).shape))

**Python note.** `nn.Parameter` marks a tensor as a weight that training changes, here the class token and the positions. `nn.ModuleList` holds the blocks so that the network knows their weights. In `forward`, `self.embed(x)` gives 64 numbers for each of the 8 × 8 squares; `.flatten(2)` puts the squares in a row and `.transpose(1, 2)` makes every square a token of 64 numbers. `encode` puts the class token before the tokens with `torch.cat`, adds the positions, passes all through the blocks and reads the class token at place 0.

**A ready vision transformer from the Hugging Face Hub.** *Lecture: [Building a ViT with PyTorch; a ready ViT model from Hugging Face](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#building-a-vit-with-pytorch-a-ready-vit-model-from-hugging-face)*

The Hugging Face Hub keeps ViT-B/16 as Google trained it, under the name "google/vit-base-patch16-224": pretrained on ImageNet-21k and fine-tuned on the 1,000 classes of ImageNet. The cell loads its plan, its preparation of images and its weights, about 350 MB, which are downloaded once. Where the Hub cannot be reached, the cell builds the same plan with random weights and says so: The structure and the shapes are then the same, the numbers are not.

**From the formulas to the code.** The plan: `ViTConfig.from_pretrained(HF_NAME)`. The network with its weights: `ViTForImageClassification.from_pretrained(HF_NAME)`. The attention of every layer: `output_attentions=True`.

In [ ]:
from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor

HF_NAME = "google/vit-base-patch16-224"
hf_config = ViTConfig.from_pretrained(HF_NAME, attn_implementation="eager")   # a small text file: the plan of the network
hf_processor = ViTImageProcessor.from_pretrained(HF_NAME)  # how an image is prepared for the network
try:
    hf_vit = ViTForImageClassification.from_pretrained(HF_NAME, attn_implementation="eager").eval()   # the weights: about 350 MB, downloaded once
    hf_pretrained = True
except Exception:
    hf_vit = ViTForImageClassification(hf_config).eval()                 # the same plan with random weights
    hf_pretrained = False

print("weights from the Hub:", hf_pretrained)
print("image", hf_config.image_size, "· patch", hf_config.patch_size, "· layers", hf_config.num_hidden_layers,
      "· heads", hf_config.num_attention_heads, "· numbers per token", hf_config.hidden_size,
      "· classes", len(hf_config.id2label))
print("weights:", f"{sum(p.numel() for p in hf_vit.parameters()):,}", "· mean and standard deviation of the preparation:",
      hf_processor.image_mean, hf_processor.image_std)
inputs = hf_processor(images=np.repeat((small[..., None] * 255).astype(np.uint8), 3, axis=2), return_tensors="pt")
with torch.no_grad():
    out = hf_vit(**inputs, output_attentions=True)
print("input", tuple(inputs["pixel_values"].shape), "· scores", tuple(out.logits.shape),
      "· attention of the 12 layers", len(out.attentions), "×", tuple(out.attentions[0].shape))
if hf_pretrained:
    print("ImageNet class with the highest score for the angiogram:", hf_config.id2label[int(out.logits.argmax())])

**Python note.** `from_pretrained` looks for the files of a model on the Hub by its name, downloads them once and keeps them in a folder of the computer. `try` and `except` catch the error of a missing connection, and the cell goes on with `ViTForImageClassification(hf_config)`, the same network with random weights. The image processor of this model scales the three channels to the mean 0.5 and the standard deviation 0.5, not to those of ImageNet that `to_imagenet` uses for the network of torchvision. `np.repeat` gives the grey angiogram three equal colour channels.

**Observe and reflect.** The network of torchvision above and the network of the Hub have the same plan and the same number of weights, but different weights. Where do the two come from, and why must the preparation of an image follow the network that will read it?

## 4. A quality gate for echocardiography

*Lecture: [Transfer to echocardiography: A quality gate before the automatic EF](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#transfer-to-echocardiography-a-quality-gate)*

The 600 end-diastolic frames of the 300 patients of CAMUS become features in three ways: four image statistics, the 1,280 features of EfficientNet-B0 and the 768 numbers of the class token of ViT-B/16. A logistic regression on each learns which frames the experts rated as poor; its regularisation is chosen on the validation frames, and the test AUC comes with a 95 percent interval from 2,000 bootstrap samples of the test patients.

**From the formulas to the code.** The target is 1 for a poor frame: `poor`. The gate: `fit_gate(features, poor)`. The interval: `np.percentile(bootstrap_auc(...), [2.5, 97.5])`, with the patients as the groups.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(4, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
effnet.classifier = nn.Identity()      # without its last layer, the network returns the mean of its 1280 channels


@torch.no_grad()
def effnet_features(images, batch=16):
    """The 1280 features of EfficientNet-B0 for every image (Week 4)."""
    return torch.cat([effnet(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()

camus_table = camus.table()
echo, echo_patient, echo_view, echo_quality, echo_split = [], [], [], [], []
for _, row in camus_table.iterrows():
    for view in ("2CH", "4CH"):
        img, _, _ = camus.load(row["patient"], view, "ED")
        echo.append(np.asarray(Image.fromarray(img).resize((224, 224), Image.BILINEAR)) / 255)
        echo_patient.append(row["patient"])
        echo_view.append(view)
        echo_quality.append(row["quality_" + view.lower()])
        echo_split.append(row["split"])
echo = np.clip(np.stack(echo), 0, 1).astype(np.float32)
echo_patient, echo_view, echo_quality, echo_split = map(np.array, (echo_patient, echo_view, echo_quality, echo_split))
etr, eva, ete = echo_split == "training", echo_split == "validation", echo_split == "testing"
poor = (echo_quality == "Poor").astype(int)


def image_statistics(images):
    """Four numbers per image: mean grey value, standard deviation, mean difference between neighbouring pixels and share of pixels brighter than half of the maximum."""
    return np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in images])


def fit_gate(features, target):
    """A logistic regression on standardised features; the inverse regularisation strength is chosen on the validation frames."""
    best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(features[etr], target[etr])
        a = roc_auc_score(target[eva], model.predict_proba(features[eva])[:, 1])
        if best is None or a > best[0]:
            best = (a, C, model)
    return best[2], best[1], best[0]


def bootstrap_auc(target, prob, groups, n=2000, seed=2026):
    """AUC in 2000 bootstrap samples that draw groups (patients or images) with replacement."""
    rng = np.random.default_rng(seed)
    names = np.unique(groups)
    rows = {g: np.where(groups == g)[0] for g in names}
    out = []
    for _ in range(n):
        idx = np.concatenate([rows[g] for g in rng.choice(names, len(names))])
        if 0 < target[idx].sum() < len(idx):
            out.append(roc_auc_score(target[idx], prob[idx]))
    return np.array(out)

print(len(echo), "frames")
print(pd.crosstab(echo_split, echo_quality))
features = {"statistics": image_statistics(echo), "EfficientNet-B0": effnet_features(echo), "ViT-B/16": vit_features(echo)}
for name, X in features.items():
    print(name, X.shape)

**Python note.** `camus.table()` reads the Info files of the 300 patients, and the loop loads the end-diastolic frame of both views with `camus.load`, shrinks it to 224 × 224 pixels and divides it by 255. `np.stack` puts the 600 frames into one array of 600 × 224 × 224, and `map(np.array, ...)` turns four lists into arrays at once. `pd.crosstab` counts the frames of each quality in each set. The features of ViT-B/16 take the longest, one to two minutes on an ordinary laptop.

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4))
probs = {}
for name, X in features.items():
    model, C_, va = fit_gate(X, poor)
    probs[name] = model.predict_proba(X[ete])[:, 1]
    boot = bootstrap_auc(poor[ete], probs[name], echo_patient[ete])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{name:16} C={C_:<6} validation AUC {va:.3f}  test AUC {roc_auc_score(poor[ete], probs[name]):.3f} ({lo:.3f}-{hi:.3f})")
    fpr, tpr, _ = roc_curve(poor[ete], probs[name])
    ax.plot(fpr, tpr, label=name)
ax.plot([0, 1], [0, 1], ":", color="grey")
ax.set_xlabel("1 − specificity")
ax.set_ylabel("sensitivity")
ax.legend()
plt.show()

**Python note.** `fit_gate` tries four inverse regularisation strengths, fits a logistic regression on the standardised features of the training frames with each, and keeps the one with the largest AUC on the validation frames; `make_pipeline(StandardScaler(), LogisticRegression(...))` keeps the standardisation and the model together, so that the test frames are scaled with the means of the training frames. `bootstrap_auc` draws the 50 test patients with replacement 2,000 times and keeps both frames of every patient drawn; `np.percentile` reads the 2.5th and 97.5th percentiles of the 2,000 AUCs.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
names = np.unique(echo_patient[ete])
rows_of = {p: np.where(echo_patient[ete] == p)[0] for p in names}
y_te, pv, pe = poor[ete], probs["ViT-B/16"], probs["EfficientNet-B0"]
diff = []
for _ in range(2000):
    idx = np.concatenate([rows_of[p] for p in rng.choice(names, len(names))])
    if 0 < y_te[idx].sum() < len(idx):
        diff.append(roc_auc_score(y_te[idx], pv[idx]) - roc_auc_score(y_te[idx], pe[idx]))
diff = np.array(diff)
print("ViT-B/16 minus EfficientNet-B0:", round(diff.mean(), 3), np.round(np.percentile(diff, [2.5, 97.5]), 3),
      "| share of samples above 0:", round(float((diff > 0).mean()), 3))

**Observe and reflect.** The interval of the difference contains 0. How many test patients, or how many poor frames, would you need before you could call one of the two networks the better gate?

**Exercise D.** Build the gate from the class token of ViT-B/16 for good frames against the others, with `good = (echo_quality == "Good").astype(int)`. What is its test AUC? Keep it with three decimals.

In [ ]:
data = exercise_table("D")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
good = (echo_quality == "Good").astype(int)
model, _, _ = fit_gate(features["ViT-B/16"], good)
answer = round(float(roc_auc_score(good[ete], model.predict_proba(features["ViT-B/16"][ete])[:, 1])), 3)
teach.check("Exercise D", answer)

**Mini project: a mini ViT on CIFAR-10, then transfer to EchoNet.** *Lecture: [Mini project: Training a mini ViT on CIFAR-10, then transfer to EchoNet](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#mini-project-training-a-mini-vit-on-cifar-10-then-transfer-to-echonet)*

The small vision transformer of Section 3 first learns ten classes of everyday photographs, CIFAR-10, and then learns to recognise a reduced ejection fraction from two frames of an echocardiogram. Participants with a personal registration for EchoNet-Dynamic use its videos, as in the weekly project of Week 3; the others use the echocardiograms of CAMUS. The same network trained on the echocardiograms alone, from random weights, is the comparison.

The first cell downloads CIFAR-10 once, about 170 MB from the University of Toronto. The training takes about one minute per epoch on an ordinary computer; a graphics card in Colab is not needed.

**From the formulas to the code.** The ten classes: the cross-entropy of Week 3, `nn.CrossEntropyLoss()`. The new task: `copy.deepcopy(mini_vit)` and a new last layer `nn.Linear(64, 1)`. A pair of frames as an image: `three_channels`, the end-diastolic frame, the end-systolic frame and their difference.

In [ ]:
from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]


def cifar10():
    """CIFAR-10 from its source at the University of Toronto, about 170 MB, downloaded once into data/cifar10: 50,000 training and 10,000 test images of 32 x 32 pixels with three colour channels, and their classes."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """Images of n x 32 x 32 x 3 (values 0 to 255) → tensors of n x 3 x 32 x 32, every channel standardised with the mean and the standard deviation of the training images."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Data augmentation: the group is moved by up to 4 pixels in each direction and mirrored from left to right half of the time."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """AdamW with a learning rate of 0.001 and a weight decay of 0.05 over shuffled groups of 128 images, with the cross-entropy of the ten classes."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: training loss {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """The share of images whose class gets the highest score."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """The end-diastolic and end-systolic frames of both views of the 300 patients of CAMUS as pairs of 32 x 32 pixels, with the EF of the patient."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"set": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "patient": r["patient"],
                         "pair": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """The two traced frames of EchoNet-Dynamic videos as pairs of 32 x 32 pixels, with the EF: 2,000 training, 400 validation and 400 test videos."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"set": split, "patient": d["file"], "pair": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """A pair of frames → three channels: the end-diastolic frame, the end-systolic frame and their difference, in the places of the red, green and blue of a photograph."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Adam over shuffled groups of 32 pairs with the binary cross-entropy of a reduced EF; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    cifar = cifar10()
except Exception as error:
    cifar = None
    print("CIFAR-10 could not be downloaded here:", type(error).__name__,
          "The cells of the mini project that need it say so and go on with the rest.")
if cifar is not None:
    images, labels = cifar["training"]
    print("training", images.shape, "· test", cifar["test"][0].shape, "· images per class", np.bincount(labels))
    fig, ax = plt.subplots(1, 10, figsize=(11, 1.6))
    for c, a in enumerate(ax):
        a.imshow(images[np.where(labels == c)[0][0]])
        a.set_title(CIFAR_CLASSES[c], fontsize=8)
        a.axis("off")
    plt.show()

In [ ]:
if cifar is None:
    print("CIFAR-10 is missing, so the mini ViT is not trained on photographs here.")
else:
    mean = cifar["training"][0].mean(axis=(0, 1, 2)) / 255
    std = cifar["training"][0].std(axis=(0, 1, 2)) / 255
    x_train, y_train = to_tensor(cifar["training"][0], mean, std), torch.as_tensor(cifar["training"][1])
    x_test, y_test = to_tensor(cifar["test"][0], mean, std), torch.as_tensor(cifar["test"][1])
    torch.manual_seed(RANDOM_STATE)
    mini_vit = SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3, outputs=10)
    train_cifar(mini_vit, x_train, y_train, epochs=5)
    print("accuracy on the 10,000 test images:", round(accuracy(mini_vit, x_test, y_test), 3))

**Python note.** `to_tensor` turns the photographs into the layout of PyTorch, channels first, and standardises every colour channel with the training photographs. `shift_and_mirror` moves a group of photographs by up to four pixels and mirrors it half of the time, so that the network sees a slightly different picture in every epoch. `accuracy` counts how often the class with the highest score, `argmax(1)`, is the right one; one class in ten would be right by chance.

In [ ]:
try:
    echo_source = echonet.find()
except FileNotFoundError:
    echo_source = None
pairs = echonet_pairs(echo_source) if echo_source is not None else camus_pairs()
print("EchoNet-Dynamic" if echo_source is not None else "CAMUS", "·", len(pairs), "pairs")
X = {s: torch.as_tensor(three_channels(pairs[pairs["set"] == s]["pair"])) for s in ("TRAIN", "VAL", "TEST")}
Y = {s: torch.as_tensor((pairs[pairs["set"] == s]["EF"] < 50).to_numpy(np.float32)) for s in ("TRAIN", "VAL", "TEST")}
for s in X:
    print(s, tuple(X[s].shape), "reduced EF:", int(Y[s].sum()))
starts = {"random weights": SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3)}
if "mini_vit" in globals() and cifar is not None:
    pretrained = copy.deepcopy(mini_vit)
    pretrained.head = nn.Linear(64, 1)
    starts["CIFAR-10"] = pretrained
else:
    print("The start from CIFAR-10 needs the trained mini ViT of the cell above.")
for name, net in starts.items():
    torch.manual_seed(RANDOM_STATE)
    best = fine_tune(net, X["TRAIN"], Y["TRAIN"], X["VAL"], Y["VAL"])
    with torch.no_grad():
        p_test = torch.sigmoid(net(X["TEST"])).numpy()
    print(f"{name:20} validation AUC {best:.3f} · test AUC {roc_auc_score(Y['TEST'].numpy(), p_test):.3f}")

**Python note.** `echonet.find()` looks for the participant's own copy of EchoNet-Dynamic in data/echonet and raises `FileNotFoundError` when there is none; the cell then uses CAMUS. `pairs` is a table with one row per pair of frames. The dictionaries `X` and `Y` hold the three sets, "TRAIN", "VAL" and "TEST", with the names of EchoNet-Dynamic. `copy.deepcopy` makes an independent copy of the trained network, so that replacing its last layer leaves `mini_vit` unchanged. `fine_tune` keeps the weights of the epoch with the best validation AUC.

**Observe and reflect.** Compare the two test AUCs. Did ten classes of photographs help the network to read echocardiograms, and would you expect the answer to change with 2,000 videos of EchoNet-Dynamic instead of 400 frames of CAMUS?

**The Python path of Practice stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#last-two-hours-hybrid-cnn-vit-and-medical-imaging) and continue reading at «Last two hours: Hybrid CNN-ViT and medical imaging».

## 5. Patches of stenoses on the synthetic angiograms

*Lecture: [Synthetic and real angiograms: A model in another world](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#synthetic-and-real-angiograms-a-model-in-another-world)*

Patches of 64 × 64 pixels are cut from the angiograms of 300 training, 100 validation and 100 test patients: two around every significant stenosis, shifted at random by up to 6 pixels, and three on vessels far from every stenosis for each of them. A small convolutional network learns them for eight epochs.

**From the formulas to the code.** Each patch is standardised on its own: $(x - \bar{x})/(s + 0.02)$, `standardise`. The loss is the binary cross-entropy of the logit, `nn.BCEWithLogitsLoss()`, and `train` keeps the weights of the epoch with the best validation AUC.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(5, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def standardise(X):
    """Every patch on its own: mean grey value 0 and standard deviation 1."""
    X = np.asarray(X, dtype=np.float32)
    m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
    return (X - m) / (s + 0.02)

syn = {}
for s, n in (("training", 300), ("validation", 100), ("test", 100)):
    X, y_, ids = angiogram.patches(s, n_patients=n)
    syn[s] = (standardise(X), y_, ids)
    print(s, X.shape, "on a stenosis:", int(y_.sum()))
X_raw, y3, _ = angiogram.patches("test", n_patients=3)
fig, ax = plt.subplots(1, 6, figsize=(10, 1.9))
for a, i in zip(ax, list(np.where(y3 == 1)[0][:3]) + list(np.where(y3 == 0)[0][:3])):
    a.imshow(X_raw[i], cmap="gray")
    a.set_title("stenosis" if y3[i] else "vessel", fontsize=9)
    a.axis("off")
plt.show()

**Python note.** `angiogram.patches` draws the angiograms of the first `n_patients` patients of a set and cuts the patches; it returns the patches, their labels and the patient of each. Drawing 1,000 angiograms takes about two minutes. `syn` is a dictionary whose values are triples, and `standardise` brings every patch to a mean of 0 and a standard deviation of 1 on its own, with `axis=(1, 2)` and `keepdims=True` for one mean per patch.

In [ ]:
def conv_block(a, b):
    """Convolution of 3 x 3, batch normalisation, ReLU and max pooling of 2 x 2: half the size, b channels."""
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class SmallCNN(nn.Module):
    """Four convolution blocks (16, 32, 64 and 64 channels), the mean of every channel and one output."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



class Hybrid(SmallViT):
    """A hybrid: three convolution blocks turn the patch into 8 x 8 positions of 64 numbers, the tokens of two transformer blocks."""
    def __init__(self, dim=64, depth=2, heads=4):
        super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
        self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



@torch.no_grad()
def predict(model, X, batch=256):
    """The probability of a stenosis for every patch."""
    model.eval()
    X = torch.as_tensor(X, dtype=torch.float32)
    return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
    """Adam with a learning rate of 0.001, groups of 64 patches turned by a random quarter turn and mirrored half of the time; the weights of the epoch with the best validation AUC are kept."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            b = order[i:i + 64]
            xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
            if torch.rand(1) < 0.5:
                xb = torch.flip(xb, dims=(3,))
            loss = loss_fn(model(xb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: validation AUC {a:.3f}")
    model.load_state_dict(state)
    return best

torch.manual_seed(RANDOM_STATE)
cnn_synthetic = SmallCNN()
print("trainable weights:", n_parameters(cnn_synthetic))
train(cnn_synthetic, syn["training"][0], syn["training"][1], syn["validation"][0], syn["validation"][1], epochs=8)
p_syn = predict(cnn_synthetic, syn["test"][0])
print("test AUC, synthetic patches:", round(roc_auc_score(syn["test"][1], p_syn), 3))

**Python note.** A network is a *class*: `class SmallCNN(nn.Module)` describes its parts in `__init__` and the way through them in `forward`, and `SmallCNN()` builds one with fresh random weights; `torch.manual_seed(RANDOM_STATE)` just before it makes these weights the same in every run. `conv_block` returns the four layers of a block as one `nn.Sequential`. `train` is the loop of Week 3: shuffled groups of 64 patches, a random quarter turn with `torch.rot90` and a mirror with `torch.flip` half of the time, the loss, `backward` and a step of Adam; after every epoch the validation AUC decides, and `copy.deepcopy` keeps the best weights. `predict` returns the probability of a stenosis, the sigmoid of the logit, for every patch.

**Exercise E.** What share of the synthetic training patches lies on a significant stenosis? Keep it with three decimals.

In [ ]:
data = exercise_table("E")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(syn["training"][1].mean()), 3)
teach.check("Exercise E", answer)

## 6. ARCADE: real angiograms

*Lecture: [Synthetic and real angiograms: A model in another world](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#synthetic-and-real-angiograms-a-model-in-another-world)*

The stenosis set of ARCADE holds 1,500 X-ray coronary angiograms of 512 × 512 pixels, with the regions of the stenotic plaques outlined by experts. Patches are cut as on the synthetic angiograms; the negatives lie on pixels that look like vessels, at least 48 pixels from every outlined stenosis and from the border of the image, where the dark frame of the X-ray field lies. The network of Section 5 is tested on them without change, and the same network is then trained on ARCADE.

**From the formulas to the code.** The centre of a stenosis is the centre of its box: `arcade.annotations(split)`. Vessel pixels are those with the largest black top-hat, a filter that marks thin dark structures: `arcade.vessel_points`. The interval of the AUC draws angiograms: `bootstrap_auc(y, p, ids)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(6, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
arcade.download()
arc = {s: arcade.patches(s) for s in arcade.SPLITS}

for s in arcade.SPLITS:
    ann = arcade.annotations(s)
    print(s, len(ann), "angiograms,", sum(len(a["stenoses"]) for a in ann), "stenoses,", len(arc[s][1]), "patches")
first = arcade.annotations("test")[0]
img = arcade.load("test", first["file"])
plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray")
for s in first["stenoses"]:
    bx, by, bw, bh = s["box"]
    plt.gca().add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, color="#9B1B1B", lw=1.5))
plt.title("ARCADE, test angiogram " + str(first["id"]))
plt.axis("off")
plt.show()

**Python note.** `arcade.download()` downloads the archive of ARCADE from Zenodo once, checks it against its published checksum and keeps only the stenosis set in the folder `data/arcade`. `arcade.patches(split)` cuts the patches of all angiograms of a set and returns the patches, their labels and the number of the angiogram of each; `arc` is a dictionary of these triples for `train`, `val` and `test`. `arcade.annotations` returns one dictionary per angiogram with its file and its stenoses, each with a box (x, y, width, height). Image: ARCADE, Popov et al. 2024, CC0.

In [ ]:
ARC = {s: (standardise(X), y_, ids) for s, (X, y_, ids) in arc.items()}
p_shift = predict(cnn_synthetic, ARC["test"][0])
lo, hi = np.percentile(bootstrap_auc(ARC["test"][1], p_shift, ARC["test"][2]), [2.5, 97.5])
print(f"Network of Section 5 on ARCADE: AUC {roc_auc_score(ARC['test'][1], p_shift):.3f} ({lo:.3f}-{hi:.3f})")
torch.manual_seed(RANDOM_STATE)
cnn_arcade = SmallCNN()
train(cnn_arcade, ARC["train"][0], ARC["train"][1], ARC["val"][0], ARC["val"][1], epochs=8)
p_cnn = predict(cnn_arcade, ARC["test"][0])
lo, hi = np.percentile(bootstrap_auc(ARC["test"][1], p_cnn, ARC["test"][2]), [2.5, 97.5])
print(f"Trained on ARCADE: AUC {roc_auc_score(ARC['test'][1], p_cnn):.3f} ({lo:.3f}-{hi:.3f})")

**Observe and reflect.** The network that learned only from synthetic angiograms falls to an AUC of about 0.54 on ARCADE. Compare a synthetic patch and a patch of ARCADE: Which differences could the network have used on the synthetic images that real images do not offer?

**Advanced (optional).** Can the synthetic network help when real images are few? Fine-tune it on a tenth of the training angiograms of ARCADE and compare it with the same network started from random weights: `tenth = np.isin(ARC["train"][2], np.random.default_rng(RANDOM_STATE).choice(np.unique(ARC["train"][2]), 100, replace=False))`, then `train(copy.deepcopy(cnn_synthetic), ARC["train"][0][tenth], ARC["train"][1][tenth], ARC["val"][0], ARC["val"][1])`. In the run of the course, the test AUCs of the two networks were 0.71 and 0.67.

**Exercise F.** How many stenoses did the experts outline on average in one training angiogram of ARCADE? Keep the mean with three decimals.

In [ ]:
data = exercise_table("F")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.mean([len(a["stenoses"]) for a in arcade.annotations("train")])), 3)
teach.check("Exercise F", answer)

**The Python path of Practice stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared) and continue reading at «A convolutional network, a vision transformer and a hybrid compared».

## 7. A small vision transformer and a hybrid

*Lecture: [A convolutional network, a vision transformer and a hybrid compared](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared)*

The small vision transformer and the hybrid learn the patches of ARCADE exactly as the convolutional network of Section 6 did. The three are compared on the same test patches, with intervals from a bootstrap of the test angiograms, and at the threshold that gives a sensitivity of 0.90 on the validation patches.

**From the formulas to the code.** The tokens of the vision transformer: `self.embed(x).flatten(2).transpose(1, 2)`, 64 tokens of 64 numbers; $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})$ is `torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)` in `Attention`. The threshold for a sensitivity of 0.90: the probability that 90 percent of the validation patches with a stenosis reach.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(7, "Catch-up finished: the code of the earlier sections has run.")

**Swin Transformer: attention in shifted windows.** *Lecture: [Swin Transformer: Shifted window attention](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#swin-transformer-shifted-window-attention)*

A Swin Transformer computes attention only inside windows of M × M tokens and moves the windows by half a window in every second layer, so that information crosses the borders of the windows. The cell counts the multiplications of one layer of attention, global and in windows, with the formulas of the Swin Transformer paper, and follows on a grid of 8 × 8 tokens with windows of 4 × 4 how far information spreads from one corner, with and without the shift.

**From the formulas to the code.** $\Omega(\mathrm{MSA}) = 4hwC^2 + 2(hw)^2C$ and $\Omega(\mathrm{W\text{-}MSA}) = 4hwC^2 + 2M^2hwC$: `attention_cost(h, w, C)` and `attention_cost(h, w, C, M)`. The windows of every token: `windows(grid, size, shifted)`.

In [ ]:
def windows(grid, size, shifted=False):
    """The window of every token of a grid of grid x grid tokens: windows of size x size tokens; shifted windows start half a window further, so the windows at the edges are smaller."""
    s = size // 2 if shifted else 0
    rows, cols = np.indices((grid, grid))
    n = grid // size + (1 if shifted else 0)
    return ((rows + s) // size) * n + (cols + s) // size


def reach(grid, size, layers, start=(0, 0), shift=True):
    """How many tokens can have passed information to the token at start after each layer of window attention, with or without shifted windows in every second layer."""
    seen = np.zeros((grid, grid), bool)
    seen[start] = True
    counts = []
    for layer in range(layers):
        part = windows(grid, size, shifted=shift and layer % 2 == 1)
        seen = np.isin(part, np.unique(part[seen]))
        counts.append(int(seen.sum()))
    return counts


def attention_cost(h, w, C, M=None):
    """The multiplications of one layer of attention, as in the Swin Transformer paper: global, 4hwC² + 2(hw)²C, or in windows of M x M tokens, 4hwC² + 2M²hwC."""
    return 4 * h * w * C ** 2 + (2 * (h * w) ** 2 * C if M is None else 2 * M ** 2 * h * w * C)

for name, side in (("image of 224 x 224, patches of 4 x 4", 56), ("angiogram of 512 x 512, patches of 4 x 4", 128)):
    g, w = attention_cost(side, side, 96), attention_cost(side, side, 96, M=7)
    print(f"{name}: {side * side:,} tokens · global {g / 1e9:.2f} · windows of 7 x 7 {w / 1e9:.3f} billion multiplications · {g / w:.1f}x")
print("tokens reached after each layer, with shifted windows:", reach(8, 4, 4))
print("the same without the shift:", reach(8, 4, 4, shift=False))
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
for a, shifted, title in zip(ax, (False, True), ("layer 1: windows", "layer 2: shifted windows")):
    a.imshow(windows(8, 4, shifted), cmap="tab20")
    a.set_title(title, fontsize=9)
    a.set_xticks([])
    a.set_yticks([])
plt.show()

**Python note.** `np.indices((grid, grid))` gives the row and the column of every token. Integer division `//` by the size of a window gives the window of every token; with the shift, the count starts half a window earlier, so the windows at the edges are smaller, as the masking of the Swin Transformer makes them. `reach` marks the tokens that share a window with any token reached so far, layer after layer. Without the shift, the corner token never sees past its own window.

In [ ]:
import time
networks = {"CNN": cnn_arcade}
seconds = {}
for name, M in (("ViT", SmallViT), ("Hybrid", Hybrid)):
    torch.manual_seed(RANDOM_STATE)
    net = M()
    t0 = time.time()
    train(net, ARC["train"][0], ARC["train"][1], ARC["val"][0], ARC["val"][1], epochs=8)
    seconds[name] = round(time.time() - t0)
    networks[name] = net
vit_arcade, hybrid_arcade = networks["ViT"], networks["Hybrid"]

**Python note.** `SmallViT` puts the parts of the lecture page into code. `self.embed`, a convolution with filters of 8 × 8 and a stride of 8, applies the same linear map to every square of 8 × 8 pixels; `flatten(2).transpose(1, 2)` turns its output into 64 tokens of 64 numbers. `encode` puts the class token `self.cls` before them with `torch.cat`, adds the positions `self.pos` and passes the tokens through the blocks; the class token of the last block goes through `self.norm` and `self.head`. `Hybrid` inherits all of this from `SmallViT` and only replaces `self.embed` by three convolution blocks, whose output of 8 × 8 positions gives the tokens.

In [ ]:
yt, yv = ARC["test"][1], ARC["val"][1]
probs_te = {k: predict(m, ARC["test"][0]) for k, m in networks.items()}
probs_va = {k: predict(m, ARC["val"][0]) for k, m in networks.items()}
rows, thresholds = [], {}
for k in networks:
    p, pv = probs_te[k], probs_va[k]
    thresholds[k] = float(np.sort(pv[yv == 1])[int(np.floor(0.1 * yv.sum()))])
    hit = p >= thresholds[k]
    lo, hi = np.percentile(bootstrap_auc(yt, p, ARC["test"][2]), [2.5, 97.5])
    rows.append({"network": k, "weights": n_parameters(networks[k]), "AUC": round(roc_auc_score(yt, p), 3), "95%": f"{lo:.3f}-{hi:.3f}",
                 "sensitivity": round(hit[yt == 1].mean(), 3), "specificity": round(1 - hit[yt == 0].mean(), 3),
                 "PPV": round((hit & (yt == 1)).sum() / hit.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("training time (s):", seconds)

**Python note.** `np.sort(pv[yv == 1])` lists the probabilities of the validation patches with a stenosis from the smallest up; the value at the position of 10 percent is reached by 90 percent of them, so it is the threshold for a sensitivity of 0.90 on the validation set. The test set only applies it. `to_string(index=False)` prints the table without the numbers of its rows.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
names = np.unique(ARC["test"][2])
rows_of = {i: np.where(ARC["test"][2] == i)[0] for i in names}
diff = {"ViT": [], "Hybrid": []}
for _ in range(2000):
    idx = np.concatenate([rows_of[i] for i in rng.choice(names, len(names))])
    base = roc_auc_score(yt[idx], probs_te["CNN"][idx])
    for k in diff:
        diff[k].append(roc_auc_score(yt[idx], probs_te[k][idx]) - base)
for k, d in diff.items():
    print(f"{k} minus the convolutional network: {np.mean(d):.3f} ({np.percentile(d, 2.5):.3f}-{np.percentile(d, 97.5):.3f})")

**Observe and reflect.** On the 5,106 synthetic patches of Section 5 the convolutional network did best; on the 13,018 patches of ARCADE the order turns. Which differences between the two sets, besides their size, could explain this?

**Exercise G.** How many of the trainable weights of the small vision transformer sit in its four transformer blocks, `vit_arcade.blocks`?

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = sum(p.numel() for p in vit_arcade.blocks.parameters())
teach.check("Exercise G", answer)

## 8. Attention maps

*Lecture: [Attention map visualisation (clinical interpretation)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#attention-maps-read-clinically-with-their-limits)*

Every block of the small vision transformer keeps the attention of its last call. Followed through the four layers with the rollout, the attention of the class token gives a map of 8 × 8 squares for every patch, and the share of the map on the four central squares shows whether the network looks where a stenosis would lie.

**From the formulas to the code.** $\tilde{A}_l = \mathrm{rows}(\tfrac{1}{2}A_l + \tfrac{1}{2}I)$ and $R = \tilde{A}_L \cdots \tilde{A}_1$: `rollout(maps)` of Section 3. The attention of a block: `block.attn.weights`, with the heads along its second axis.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(8, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
@torch.no_grad()
def class_token_maps(model, X):
    """The rollout of the class token over the 64 squares of every patch: an array of n x 8 x 8."""
    model.eval()
    out = []
    for i in range(0, len(X), 256):
        model(torch.as_tensor(X[i:i + 256], dtype=torch.float32)[:, None])
        A = np.stack([b.attn.weights.mean(dim=1).numpy() for b in model.blocks], axis=1)
        out += [rollout(a)[0, 1:].reshape(8, 8) for a in A]
    return np.array(out)


maps_te = class_token_maps(vit_arcade, ARC["test"][0])
share = maps_te[:, 3:5, 3:5].sum(axis=(1, 2)) / maps_te.sum(axis=(1, 2))
print("share of the four central squares, with a stenosis:", round(share[yt == 1].mean(), 3),
      "| without:", round(share[yt == 0].mean(), 3), "| even spread:", 4 / 64)
raw_test = arc["test"][0]
pick = list(np.where(yt == 1)[0][:3]) + list(np.where(yt == 0)[0][:3])
fig, ax = plt.subplots(1, 6, figsize=(11, 2.2))
for a, i in zip(ax, pick):
    a.imshow(raw_test[i], cmap="gray")
    a.imshow(np.kron(maps_te[i], np.ones((8, 8))), cmap="inferno", alpha=0.45)
    a.set_title(f"{'stenosis' if yt[i] else 'vessel'} {probs_te['ViT'][i]:.2f}", fontsize=9)
    a.axis("off")
plt.show()

**Python note.** `b.attn.weights` holds, for the last group of patches, an array of patches × heads × 65 × 65; `.mean(dim=1)` averages the four heads, and `np.stack(..., axis=1)` puts the four layers of every patch side by side, so that `rollout(a)` can multiply them. `[0, 1:]` takes the row of the class token without its attention to itself. `maps_te[:, 3:5, 3:5]` takes the four central squares of every map, the central 16 × 16 pixels of the patch. Images: ARCADE, Popov et al. 2024, CC0.

In [ ]:
k = int(np.where((echo_patient == "patient0201") & (echo_view == "4CH"))[0][0])
fig, ax = plt.subplots(1, 2, figsize=(6.4, 3.2))
for a, (title, img_) in zip(ax, (("case, left angiogram", small), ("CAMUS, patient0201, 4CH", echo[k]))):
    r = rollout(vit_attention(img_))[0, 1:].reshape(14, 14)
    a.imshow(np.asarray(Image.fromarray((img_ * 255).astype(np.uint8)).resize((224, 224))), cmap="gray")
    a.imshow(np.kron(r, np.ones((16, 16))), cmap="inferno", alpha=0.5)
    a.set_title(title, fontsize=9)
    a.axis("off")
plt.show()

**Observe and reflect.** ViT-B/16 learned from photographs only. On the angiogram and on the echocardiogram, where does its rollout lie, and would a cardiologist look there? What would you need before you show such a map to a patient or a colleague?

**Exercise H.** What is the mean share of the rollout on the four central squares for the test patches without a stenosis? Use `maps_te` and `yt`. Keep it with three decimals.

In [ ]:
data = exercise_table("H")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s = maps_te[:, 3:5, 3:5].sum(axis=(1, 2)) / maps_te.sum(axis=(1, 2))
answer = round(float(s[yt == 0].mean()), 3)
teach.check("Exercise H", answer)

## 9. From patches to the whole angiogram

*Lecture: [From patches to the whole angiogram](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#from-patches-to-the-whole-angiogram)*

The hybrid scans a whole test angiogram with windows of 64 × 64 pixels every 32 pixels, 256 windows per angiogram; the code below does so for the first 20 test angiograms. At the threshold of the validation patches, a stenosis counts as found when a window whose centre lies within 32 pixels of it reaches the threshold, and every other window over the threshold is an alarm.

**From the formulas to the code.** The grid of probabilities: `heat_map(model, image)`. Stenoses found and alarm windows: `lesions_found(grid, centres, stenoses, threshold)`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(9, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
def heat_map(model, image, size=64, stride=32):
    """The probability of a stenosis for windows of 64 x 64 pixels every 32 pixels: a grid of 16 x 16 for an angiogram of 512 x 512."""
    h = size // 2
    pad = np.pad(image, h, mode="reflect")
    centres = np.arange(stride // 2, image.shape[0], stride)
    windows = [pad[r:r + size, c:c + size] for r in centres for c in centres]
    return predict(model, standardise(windows)).reshape(len(centres), len(centres)), centres


def lesions_found(grid, centres, stenoses, threshold, near=32):
    """A stenosis is found when a window whose centre lies within 32 pixels of it reaches the threshold; an alarm is a window over the threshold farther than 32 pixels from every stenosis."""
    cy, cx = np.meshgrid(centres, centres, indexing="ij")
    hot = grid >= threshold
    found, alarm = 0, hot.copy()
    for s in stenoses:
        close = np.hypot(cx - s["x"], cy - s["y"]) <= near
        found += int((hot & close).any())
        alarm &= ~close
    return found, int(alarm.sum())

test_ann = arcade.annotations("test")[:20]
grids = []
found = total = alarms = 0
for a in test_ann:
    grid, centres = heat_map(hybrid_arcade, arcade.load("test", a["file"]))
    grids.append(grid)
    f_, n_alarm = lesions_found(grid, centres, a["stenoses"], thresholds["Hybrid"])
    found, total, alarms = found + f_, total + len(a["stenoses"]), alarms + n_alarm
flagged = np.mean([(g >= thresholds["Hybrid"]).mean() for g in grids])
print(f"stenoses found: {found}/{total} | alarm windows per angiogram: {alarms / len(test_ann):.1f} | share of windows over the threshold: {flagged:.2f}")
a, g = test_ann[0], grids[0]
plt.figure(figsize=(4.4, 4.4))
plt.imshow(arcade.load("test", a["file"]), cmap="gray")
plt.imshow(np.kron(g, np.ones((32, 32))), cmap="inferno", alpha=0.45, vmin=0, vmax=1)
for s in a["stenoses"]:
    bx, by, bw, bh = s["box"]
    plt.gca().add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, color="white", lw=1.5))
plt.axis("off")
plt.show()

**Python note.** `heat_map` pads the image by reflecting its border, cuts a window around every point of a grid of centres, standardises the windows and asks the network for their probabilities, which it returns as a grid of 16 × 16 with the centres. `lesions_found` compares the grid with the stenoses: `np.meshgrid` gives the coordinates of every centre, `np.hypot` the distance to a stenosis, and the windows that are far from every stenosis and over the threshold are the alarms. Image: ARCADE, Popov et al. 2024, CC0.

**Observe and reflect.** Nearly every stenosis is found, and most windows raise an alarm. Which windows of a whole angiogram did the network never see in training? How would you change the patches of training before scanning again?

**Exercise I.** On the 20 test angiograms of the cell above, what share of the windows of the convolutional network of Section 6 reaches its own threshold, `thresholds["CNN"]`? Keep the mean share over the angiograms with two decimals.

In [ ]:
data = exercise_table("I")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.mean([(heat_map(cnn_arcade, arcade.load("test", a["file"]))[0] >= thresholds["CNN"]).mean() for a in test_ann])), 2)
teach.check("Exercise I", answer)

**The Python path of Practice stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#ai-pair-programming-part-7-reviewing-an-image-pipeline) and continue reading at «Working with an AI assistant, part 7: Reviewing an image pipeline». The rest of this Jupyter notebook holds further practice and the help for the weekly project.

## 10. The review of an image pipeline: a random split

*Lecture: [Working with an AI assistant, part 7: Reviewing an image pipeline](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lecture.html#ai-pair-programming-part-7-reviewing-an-image-pipeline)*

An AI assistant asked to test a model on held-out patches put a fifth of the training patches aside at random. The code below compares this split with a split by angiogram, in which a fifth of the training angiograms, with all their patches, is put aside. The model is a nearest-neighbour model: It keeps the training patches and gives a new patch, as its probability of a stenosis, the share of stenoses among the five most similar training patches. Such a model memorises, and the weakness of a split shows most clearly on it.

**From the formulas to the code.** The random split: `rng.permutation`, a fifth of the patches. The split by angiogram: `np.isin(ids, held)`, with `held` a fifth of the angiograms. The shared angiograms: the intersection of two sets, `&`.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, done, name="NB07_vision_transformers.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


_catch_up(10, "Catch-up finished: the code of the earlier sections has run.")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
Xtr, ytr, ids = ARC["train"]
rows32 = Xtr.reshape(len(Xtr), 32, 2, 32, 2).mean(axis=(2, 4)).reshape(len(Xtr), -1)
rng = np.random.default_rng(RANDOM_STATE)
held_random = np.zeros(len(ytr), dtype=bool)
held_random[rng.permutation(len(ytr))[:len(ytr) // 5]] = True
held = rng.choice(np.unique(ids), len(np.unique(ids)) // 5, replace=False)
held_image = np.isin(ids, held)
for name, out in (("random", held_random), ("by angiogram", held_image)):
    shared = set(ids[out]) & set(ids[~out])
    knn = KNeighborsClassifier(n_neighbors=5).fit(rows32[~out], ytr[~out])
    auc = roc_auc_score(ytr[out], knn.predict_proba(rows32[out])[:, 1])
    print(f"{name:16} angiograms on both sides: {len(shared):4} | AUC on the held-out patches: {auc:.3f}")

**Python note.** `reshape(len(Xtr), 32, 2, 32, 2).mean(axis=(2, 4))` shrinks every patch to 32 × 32 pixels by averaging squares of 2 × 2 pixels, and the last `reshape` turns it into a row of 1,024 numbers. `KNeighborsClassifier(n_neighbors=5)` keeps the training rows in `fit`; `predict_proba` finds for every new row the five closest training rows and returns the share of each class among them. `np.zeros(..., dtype=bool)` makes a mask of `False` values, and the first fifth of a random permutation of the rows becomes `True`. `rng.choice(..., replace=False)` draws a fifth of the angiograms without repeating any, and `np.isin` marks every patch of these angiograms. `set(...) & set(...)` keeps the angiograms that occur on both sides.

**Observe and reflect.** With the random split, the nearest neighbour of a held-out patch with a stenosis is often its twin, the same stenosis shifted by a few pixels. A small convolutional network trained for a few epochs with augmentation memorises less than this model, and a large network trained for long memorises more. Why does the split by angiogram, or by patient, stay necessary whatever the model?

**Exercise J.** How many training angiograms of ARCADE have patches on both sides of the random split, `held_random`?

In [ ]:
data = exercise_table("J")   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(set(ids[held_random]) & set(ids[~held_random]))
teach.check("Exercise J", answer)

## Application challenges by field

| Field | Challenge |
|---|---|
| Interventional cardiology | Draw the synthetic angiograms of the test patients with a stenosis of 50 to 69 percent in the LAD. Does the hybrid, trained on ARCADE, give them higher probabilities than the patches of significant stenoses? What would this mean for a tool in the laboratory? |
| Echocardiography | Build the quality gate of Section 4 for the two-chamber and the four-chamber views separately. Is a poor four-chamber frame harder to flag than a poor two-chamber frame, and how many poor frames of each view does the test set hold? |
| Heart failure | Join the quality gate to the automatic ejection fraction of Week 4: Is the difference between the network and the expert larger for the frames that the gate flags? Use the test patients of CAMUS. |
| Cardiac imaging | Train the small vision transformer with squares of 16 × 16 pixels instead of 8 × 8 (16 tokens instead of 64). How do the number of weights, the training time and the test AUC change? |
| Clinical research | Repeat the training of the hybrid with five random seeds. How large is the spread of its test AUC, and is the difference to the vision transformer larger than this spread? |
| Quality and safety | Choose ten test patches of ARCADE that the hybrid gets wrong with a probability over 0.9 or under 0.1, and look at them with their attention maps. Which of the errors could a cardiologist explain from the image? |
| Medical informatics | Write the checklist that a hospital should apply before a vendor's tool for stenoses runs on its angiograms: data, reference, sets, metrics at the level of the decision, privacy and the reading of the alarms. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-07/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of this week extends Sections 7 and 9: The three networks are compared on ARCADE, their thresholds are judged at two prevalences, and the errors of a detector for whole angiograms are read by a cardiologist.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>